# 01 · FTIR Microplastics — Raw Data Exploration

**Texas A&M University · Department of Geology & Geophysics · FTIR Microplastics Research Platform**

This notebook is a **discovery and feasibility study** of our raw FTIR microplastics data. It answers three questions before we build the production pipeline (S3 Bronze/Silver/Gold → RDS PostgreSQL → Databricks):

1. **What do we actually have?** Which folders, formats, file sizes, and duplicates.
2. **Can we read it reliably?** Which files contain spectra, which contain metadata, and which cannot be read yet.
3. **What will the pipeline need to handle?** Data-quality issues, missing identifiers, and preprocessing requirements.

### Ground rules this notebook follows

| Rule | How it is enforced |
|---|---|
| Source data is **read-only** | Files are only opened in read mode. The notebook refuses to run if the output folder is inside the data folder. |
| **Nothing is invented** | No synthetic spectra and no guessed polymer labels. Any value inferred by a heuristic is labelled *heuristic*. |
| **Nothing is silently dropped** | Every file gets a parse status, and every problem is logged to `ftir_eda_outputs/logs/ftir_eda.log` and `tables/qc_issues.csv`. |
| **No destructive preprocessing** | Baseline correction, smoothing, and similar steps are only *recommended* here. Display-only scaling is labelled on every plot that uses it. |
| **Reproducible** | Random sampling uses a fixed seed, and re-running overwrites outputs instead of duplicating them. |

### How to use it

1. Run **Section 1** and set `DATA_ROOT` (instructions below).
2. Then use **Runtime → Run all** (Colab) or **Run All** (VS Code).
3. Cells marked **⚠️ REQUIRES DATA** print a clear message and skip themselves when the data folder cannot be found, so the notebook never crashes because data is missing.

## Notebook outline and file-loading strategy

| § | Section | What it produces |
|---|---|---|
| 1 | Environment & Google Drive access | Configuration, read-only guard, output folders, log file |
| 2 | Dataset inventory ⚠️ | Directory tree, file counts, formats, sizes, duplicate names, SHA-256 checksums |
| 3 | Initial dataset inspection ⚠️ | Columns, head rows, dimensions, data types, missing values, duplicate rows, units, for a seeded sample of files from each category |
| 4 | FTIR spectral data identification ⚠️ | Spectra found in every readable file, plus axis checks (range, ordering, spacing, duplicates, non-finite values, units) |
| 5 | Initial spectral visualizations ⚠️ | Representative spectrum, overlays, range/length distributions, missingness, intensity summaries, label comparison (only if labels exist) |
| 6 | Metadata exploration ⚠️ | Metadata field catalogue, PostgreSQL type candidates, spectrum ↔ metadata ID linkage report |
| 7 | Data-quality assessment ⚠️ | Issue register that separates **confirmed errors** from **potential scientific anomalies** |
| 8 | Preprocessing recommendations | Evidence-based candidates with rationale, risks, and conditions (nothing applied) |
| 9 | Data-architecture readiness | Bronze manifest draft, PostgreSQL DDL draft, raw→processed lineage spec |
| 10 | Final research summary | `reports/EDA_SUMMARY.md` and `reports/eda_summary.json` |

**File-loading strategy**

* **Inventory first, parse second.** Section 2 only lists files and computes checksums. It never interprets file contents, so even an unreadable file gets counted.
* **Format detection uses content, not just extensions.** Before any text parsing, the first bytes of each file are checked for binary content. Binary data is never read as text.
* **Text tables are sniffed.** For CSV/TSV/TXT/DAT files, the notebook detects the delimiter (comma, tab, semicolon, pipe, whitespace), the decimal convention (`.` or `,`), any instrument header ("preamble") lines, and the column-header row. Preamble lines are parsed as `key: value` metadata.
* **Spreadsheets** are read sheet by sheet. The header row is detected, because title rows above the table are common.
* **Vendor formats** use dedicated readers that are installed *only if those files are present*:

  | Format | Reader | Reliability |
  |---|---|---|
  | JCAMP-DX (`.jdx`, `.dx`) | `jcamp` | Good. This is an open text standard. |
  | Galactic/Thermo SPC (`.spc`) | `spc-io` | Good for the modern SPC format (0x4B/0x4C). The old 0x4D format may fail. |
  | Bruker OPUS (`.0`, `.1`, …) | `brukeropusreader` | Reads the `AB` block. Verify against OPUS for a few files. |
  | Thermo OMNIC (`.spa`, `.spg`) | `spectrochempy` (optional, large install) | Reasonable, but turned off by default (`ENABLE_SPA_READER`). An alternative is to export CSV from OMNIC. |

* **Supported spectral layouts:** two-column x/y files, *wide-by-columns* (one wavenumber column and one column per spectrum), *wide-by-rows* (wavenumbers as column headers and one row per spectrum), and *long* format (sample ID, wavenumber, value).
* **Memory-safe.** Files are processed one at a time, and only summary statistics are kept for every spectrum. A fixed-size, seeded random sample of spectra (reservoir sampling) is kept for plotting.

---
## 1 · Environment and Google Drive access

### 1.1 Libraries
This notebook needs only `numpy`, `pandas`, `matplotlib`, and `openpyxl` (for `.xlsx`), all of which are pre-installed in Colab. Format-specific readers are installed later, **only if** matching files are found.

In [ ]:
import csv
import hashlib
import importlib
import json
import logging
import os
import platform
import re
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    from tqdm.auto import tqdm
except ImportError:  # progress bars are optional
    def tqdm(iterable, **kwargs):
        return iterable

IN_COLAB = "google.colab" in sys.modules

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
})

print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")
print("Environment:", "Google Colab" if IN_COLAB else "local Jupyter / VS Code")

### 1.2 Access the data

**Important: how Google Drive access works**

* `drive.mount()` opens Google's own sign-in window. The notebook can then see **only the files that the signed-in account can access**. No password or token is stored in this notebook, so never paste credentials into a cell.
* A shared-folder URL (`https://drive.google.com/drive/folders/1M_8YRX2...`) is **not** a file path. Folders that were *shared with you* do **not** appear under `/content/drive` automatically.

**Option A (recommended in Colab): add a shortcut**
1. Open the shared folder in Google Drive in your browser.
2. Right-click the folder name → **Organize → Add shortcut** → choose **My Drive**.
3. After mounting, the folder appears at `/content/drive/MyDrive/<folder name>`. Set `DATA_ROOT` to that path in the next code cell.
4. If the folder is on a *Shared drive*, it appears under `/content/drive/Shareddrives/<drive name>/...`.

**Option B (local VS Code or offline): download a copy**
1. In Drive, right-click the folder → **Download**, then unzip it.
2. Set `DATA_ROOT` to the unzipped folder, or set the `FTIR_DATA_ROOT` environment variable before starting Jupyter.

In [ ]:
MOUNT_GOOGLE_DRIVE = True  # set to False if you are using a downloaded copy of the data

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    # Google's sign-in prompt appears here. No credentials are written to this notebook.
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not running in Colab, or MOUNT_GOOGLE_DRIVE = False).")

### 1.3 Configuration
Edit **only this cell** to point at the data and to change sampling limits. Every other cell reads these values.

In [ ]:
# ---- Raw data location (opened READ-ONLY) ----------------------------------------------
# The FTIR_DATA_ROOT environment variable overrides the defaults below.
if os.environ.get("FTIR_DATA_ROOT"):
    DATA_ROOT = Path(os.environ["FTIR_DATA_ROOT"])
elif IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/FTIR_Microplastics_Data")  # <- edit to match your shortcut's name
else:
    DATA_ROOT = Path("data/raw")                                         # <- edit to your downloaded copy

# ---- Output location (must be OUTSIDE DATA_ROOT) ---------------------------------------
if os.environ.get("FTIR_OUTPUT_DIR"):
    OUTPUT_DIR = Path(os.environ["FTIR_OUTPUT_DIR"])
elif IN_COLAB and Path("/content/drive/MyDrive").is_dir():
    OUTPUT_DIR = Path("/content/drive/MyDrive/ftir_eda_outputs")  # kept after the Colab session ends
else:
    OUTPUT_DIR = Path("ftir_eda_outputs")

# ---- Reproducibility and resource limits -----------------------------------------------
RANDOM_SEED = 42
SAMPLE_FILES_PER_CATEGORY = 3    # files shown in detail per category in Section 3
MAX_FILES_TO_PARSE = None        # None = scan every candidate file; set an integer for a seeded random subset
MAX_SPECTRA_FOR_PLOTS = 300      # spectra kept in memory for plotting (reservoir sample)
MAX_TEXT_FILE_MB = 200           # larger text files are logged as "skipped_too_large", never dropped silently
MAX_METADATA_TABLE_ROWS = 200_000
TEXT_SNIFF_BYTES = 64_000        # bytes read to detect delimiter, decimal mark and header lines
HASH_ALL_FILES = True            # SHA-256 for every file (needed for the Bronze manifest); False = only same-size files
INSTALL_OPTIONAL_READERS = True  # pip-install format readers only when matching files exist
ENABLE_SPA_READER = False        # SpectroChemPy is a large install; enable only if .spa/.spg files are present
CLEAN_PREVIOUS_OUTPUTS = True    # delete outputs from the previous run, so results are never duplicated

# Used only to RECOGNISE a spectral axis, never to filter data.
WAVENUMBER_DETECT_RANGE = (200.0, 15000.0)  # cm-1 (mid-IR ~400-4000, near-IR up to ~12500)

In [ ]:
DATA_ROOT = DATA_ROOT.expanduser().resolve()
OUTPUT_DIR = OUTPUT_DIR.expanduser().resolve()
DATA_AVAILABLE = DATA_ROOT.is_dir()


def is_within(path, parent):
    try:
        path.relative_to(parent)
        return True
    except ValueError:
        return False


if DATA_AVAILABLE and is_within(OUTPUT_DIR, DATA_ROOT):
    raise ValueError(f"OUTPUT_DIR ({OUTPUT_DIR}) is inside DATA_ROOT ({DATA_ROOT}). "
                     "Choose an output folder outside the raw data so source files stay untouched.")

OUT = {name: OUTPUT_DIR / name for name in ("tables", "figures", "reports", "logs")}
for _d in OUT.values():
    _d.mkdir(parents=True, exist_ok=True)

# Re-running replaces outputs instead of piling up stale ones. Only files inside OUTPUT_DIR are touched.
GENERATED_SUFFIXES = {".csv", ".png", ".md", ".json", ".txt", ".log", ".sql"}
if CLEAN_PREVIOUS_OUTPUTS:
    _removed = 0
    for _d in OUT.values():
        for _f in _d.iterdir():
            if _f.is_file() and _f.suffix in GENERATED_SUFFIXES:
                _f.unlink()
                _removed += 1
    print(f"Removed {_removed} output file(s) from a previous run.")

# Logging: full detail goes to the log file; only warnings and errors are printed in the notebook.
logger = logging.getLogger("ftir_eda")
logger.setLevel(logging.INFO)
logger.propagate = False
for _h in list(logger.handlers):  # avoid duplicate handlers when the cell is re-run
    logger.removeHandler(_h)
    _h.close()
_fh = logging.FileHandler(OUT["logs"] / "ftir_eda.log", mode="w", encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
_sh = logging.StreamHandler(sys.stdout)
_sh.setLevel(logging.WARNING)
_sh.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(_fh)
logger.addHandler(_sh)
logger.info(f"Run started. DATA_ROOT={DATA_ROOT} OUTPUT_DIR={OUTPUT_DIR} SEED={RANDOM_SEED}")


def require_data(section):
    """Return True if the data folder is available; otherwise explain why this cell is skipped."""
    if DATA_AVAILABLE:
        return True
    display(Markdown(f"> ⚠️ **{section} skipped:** `DATA_ROOT` (`{DATA_ROOT}`) was not found. "
                     "Follow Section 1.2, update `DATA_ROOT`, and re-run."))
    return False


def seeded_rng():
    """A fresh generator per use, so results do not depend on cell execution order."""
    return np.random.default_rng(RANDOM_SEED)


def save_table(df, name, index=False):
    path = OUT["tables"] / f"{name}.csv"
    df.to_csv(path, index=index)
    return path


def save_fig(fig, name):
    path = OUT["figures"] / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return path


def hboxplot(ax, data):
    """Horizontal boxplot that works on both old and new Matplotlib versions."""
    try:
        return ax.boxplot(data, orientation="horizontal")
    except TypeError:  # Matplotlib < 3.10
        return ax.boxplot(data, vert=False)


def human_size(n):
    if n is None or (isinstance(n, float) and np.isnan(n)):
        return "n/a"
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:,.1f} {unit}"
        n /= 1024
    return f"{n:,.1f} PB"


class ReaderUnavailable(Exception):
    """A format-specific reader package could not be imported or installed."""


class ReaderNotEnabled(Exception):
    """A reader exists but is switched off in the configuration."""


def ensure_package(import_name, pip_name=None):
    """Import an optional reader, installing it with pip only if allowed. Returns the module or None."""
    try:
        return importlib.import_module(import_name)
    except ImportError:
        pass
    if not INSTALL_OPTIONAL_READERS:
        logger.warning(f"Optional package '{import_name}' is not installed and INSTALL_OPTIONAL_READERS=False.")
        return None
    try:
        print(f"Installing optional reader: {pip_name or import_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])
        importlib.invalidate_caches()
        return importlib.import_module(import_name)
    except Exception as exc:  # network errors, build failures, etc.
        logger.error(f"Could not install/import '{pip_name or import_name}': {exc}")
        return None


print(f"DATA_ROOT : {DATA_ROOT}  ->  {'FOUND' if DATA_AVAILABLE else 'NOT FOUND'}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
if not DATA_AVAILABLE:
    print("\nThe data folder was not found. Sections marked ⚠️ REQUIRES DATA will be skipped.")
    _mydrive = Path("/content/drive/MyDrive")
    if _mydrive.is_dir():
        _folders = sorted(p.name for p in _mydrive.iterdir() if p.is_dir())
        print("Folders visible at the top of My Drive (one may be your shortcut):")
        for _name in _folders[:40]:
            print("   -", _name)

---
## 2 · Dataset inventory  ⚠️ REQUIRES DATA

This section walks every folder under `DATA_ROOT` and records each file's path, extension, size, and modification time. It then computes a SHA-256 checksum, a fingerprint of the file's exact bytes. Two files with the same checksum are byte-for-byte identical, whatever their names.

Each file is assigned a **preliminary role** (spectral, metadata, image, …) based on its extension and filename. Section 4 confirms the role by reading the file's contents.

In [ ]:
EXTENSION_CATEGORIES = {
    ".csv": "delimited_text", ".tsv": "delimited_text", ".txt": "delimited_text", ".dat": "delimited_text",
    ".prn": "delimited_text", ".asc": "delimited_text", ".xy": "delimited_text", ".dpt": "delimited_text",
    ".xlsx": "spreadsheet", ".xlsm": "spreadsheet", ".xls": "spreadsheet",
    ".jdx": "jcamp_dx", ".dx": "jcamp_dx", ".jcm": "jcamp_dx",
    ".spc": "spc_binary",
    ".spa": "omnic_binary", ".spg": "omnic_binary",
    ".json": "structured_text", ".xml": "structured_text",
    ".parquet": "columnar",
    ".png": "image", ".jpg": "image", ".jpeg": "image", ".tif": "image", ".tiff": "image", ".bmp": "image",
    ".pdf": "document", ".doc": "document", ".docx": "document", ".ppt": "document", ".pptx": "document",
    ".md": "document", ".rtf": "document",
    ".zip": "archive", ".7z": "archive", ".rar": "archive", ".tar": "archive", ".gz": "archive", ".tar.gz": "archive",
}
SPECTRAL_CANDIDATE_CATEGORIES = {"delimited_text", "spreadsheet", "jcamp_dx", "spc_binary", "omnic_binary", "bruker_opus"}
SYSTEM_FILE_RE = re.compile(r"^(\.|~\$)|^(desktop\.ini|thumbs\.db)$", re.I)
METADATA_NAME_RE = re.compile(r"meta|sample|inventory|catalog|log|key|info|site|location|station|label|summary|"
                              r"manifest|record|field|sheet|list|readme", re.I)
OPUS_MAGIC = b"\x0a\x0a\xfe\xfe"  # first four bytes of Bruker OPUS files


def get_extension(name):
    lower = name.lower()
    for compound in (".tar.gz", ".csv.gz", ".txt.gz"):
        if lower.endswith(compound):
            return compound
    return Path(lower).suffix


def read_magic(path, n=8):
    with open(path, "rb") as fh:
        return fh.read(n)


def categorize(path, ext):
    """Category from the extension; Bruker OPUS (numeric extensions such as .0, .1) is confirmed by magic bytes."""
    if re.fullmatch(r"\.\d+", ext) or ext == "":
        try:
            if read_magic(path, 4) == OPUS_MAGIC:
                return "bruker_opus"
        except OSError:
            pass
        return "no_extension" if ext == "" else "unknown"
    return EXTENSION_CATEGORIES.get(ext, "unknown")


def preliminary_role(category, is_system, name_suggests_metadata):
    if is_system:
        return "system/temporary file"
    if category in {"spc_binary", "omnic_binary", "bruker_opus"}:
        return "spectral (vendor binary)"
    if category == "jcamp_dx":
        return "spectral (JCAMP-DX)"
    if category in {"delimited_text", "spreadsheet"}:
        return "metadata (by name)" if name_suggests_metadata else "spectral or tabular (confirmed in Section 4)"
    return {"image": "image (e.g., particle micrograph)", "document": "documentation",
            "archive": "archive (not extracted)", "structured_text": "structured text",
            "columnar": "columnar data"}.get(category, "unknown")


def build_inventory(root):
    """Walk the tree without opening files (except 4 magic bytes for extensionless files)."""
    rows, folders = [], []

    def on_error(err):
        logger.error(f"Cannot list directory: {err}")

    for dirpath, dirnames, filenames in os.walk(root, onerror=on_error):
        dirnames.sort()
        folder_rel = Path(dirpath).relative_to(root).as_posix()
        folders.append({"folder": folder_rel, "n_subfolders": len(dirnames), "n_files_direct": len(filenames)})
        for fname in sorted(filenames):
            path = Path(dirpath) / fname
            ext = get_extension(fname)
            is_system = bool(SYSTEM_FILE_RE.search(fname))
            meta_hint = bool(METADATA_NAME_RE.search(fname))
            row = {"rel_path": path.relative_to(root).as_posix(), "folder": folder_rel, "file_name": fname,
                   "extension": ext or "(none)", "category": None, "preliminary_role": None,
                   "size_bytes": np.nan, "modified_utc": None, "is_system_file": is_system,
                   "name_suggests_metadata": meta_hint, "stat_error": None}
            try:
                st = path.stat()
                row["size_bytes"] = st.st_size
                row["modified_utc"] = datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(timespec="seconds")
            except OSError as exc:
                row["stat_error"] = str(exc)
                logger.error(f"Cannot stat {row['rel_path']}: {exc}")
            row["category"] = categorize(path, ext)
            row["preliminary_role"] = preliminary_role(row["category"], is_system, meta_hint)
            rows.append(row)
    return pd.DataFrame(rows), pd.DataFrame(folders)


def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
inventory, folders = pd.DataFrame(), pd.DataFrame()
if require_data("Section 2 inventory"):
    inventory, folders = build_inventory(DATA_ROOT)
    inventory["sha256"] = None
    if len(inventory):
        same_size = inventory.duplicated("size_bytes", keep=False) & (inventory["size_bytes"] > 0)
        to_hash = inventory.index if HASH_ALL_FILES else inventory.index[same_size]
        for idx in tqdm(to_hash, desc="SHA-256 checksums"):
            try:
                inventory.at[idx, "sha256"] = sha256_file(DATA_ROOT / inventory.at[idx, "rel_path"])
            except OSError as exc:
                logger.error(f"Checksum failed for {inventory.at[idx, 'rel_path']}: {exc}")
    save_table(inventory, "file_inventory")
    save_table(folders, "folder_structure")

    total_bytes = inventory["size_bytes"].sum() if len(inventory) else 0
    print(f"Files found: {len(inventory):,}  |  folders: {len(folders):,}  |  total size: {human_size(total_bytes)}")
    print(f"System/temporary files (excluded from parsing, still listed): {int(inventory['is_system_file'].sum()) if len(inventory) else 0}")
    print(f"Zero-byte files: {int((inventory['size_bytes'] == 0).sum()) if len(inventory) else 0}")

### 2.1 Directory structure and files per folder

In [ ]:
def render_tree(folders_df, inventory_df, max_lines=200):
    """Text tree of folders with the number of files and total size directly inside each."""
    stats = inventory_df.groupby("folder").agg(n=("rel_path", "size"), size=("size_bytes", "sum")) if len(inventory_df) else pd.DataFrame()
    lines = [f"{DATA_ROOT.name}/"]
    for folder in sorted(folders_df["folder"]):
        if folder == ".":
            continue
        depth = folder.count("/") + 1
        n = int(stats.loc[folder, "n"]) if folder in stats.index else 0
        size = stats.loc[folder, "size"] if folder in stats.index else 0
        lines.append(f"{'    ' * depth}{folder.split('/')[-1]}/  ({n} files, {human_size(size)})")
    root_n = int(stats.loc[".", "n"]) if "." in stats.index else 0
    lines.insert(1, f"    [files directly in root: {root_n}]")
    if len(lines) > max_lines:
        lines = lines[:max_lines] + [f"... ({len(lines) - max_lines} more folders; see tables/folder_structure.csv)"]
    return "\n".join(lines)


if require_data("Section 2.1") and len(inventory):
    tree_text = render_tree(folders, inventory)
    (OUT["reports"] / "directory_tree.txt").write_text(tree_text, encoding="utf-8")
    print(tree_text)

    folder_summary = (inventory.groupby("folder")
                      .agg(n_files=("rel_path", "size"), total_bytes=("size_bytes", "sum"),
                           extensions=("extension", lambda s: ", ".join(f"{k}({v})" for k, v in Counter(s).most_common())))
                      .sort_values("n_files", ascending=False))
    save_table(folder_summary, "files_per_folder", index=True)
    display(folder_summary.head(30))

### 2.2 File formats, sizes and preliminary roles

In [ ]:
if require_data("Section 2.2") and len(inventory):
    ext_summary = (inventory.groupby(["extension", "category"])
                   .agg(n_files=("rel_path", "size"), total_bytes=("size_bytes", "sum"),
                        median_bytes=("size_bytes", "median"), min_bytes=("size_bytes", "min"),
                        max_bytes=("size_bytes", "max"))
                   .reset_index().sort_values("n_files", ascending=False))
    ext_summary["total_size"] = ext_summary["total_bytes"].map(human_size)
    save_table(ext_summary, "extension_summary")
    display(ext_summary)

    role_summary = inventory["preliminary_role"].value_counts().rename_axis("preliminary_role").reset_index(name="n_files")
    save_table(role_summary, "preliminary_roles")
    display(role_summary)

    print("Potential METADATA files (by extension and filename; confirmed in Sections 4 and 6):")
    display(inventory.loc[inventory["preliminary_role"] == "metadata (by name)", ["rel_path", "extension", "size_bytes"]].head(50))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    counts = inventory["extension"].value_counts().head(20)
    axes[0].barh(counts.index[::-1], counts.values[::-1], color="#4C72B0")
    axes[0].set_xlabel("Number of files")
    axes[0].set_title("Files per extension (top 20)")
    sized = inventory[inventory["size_bytes"] > 0]
    cats = sized["category"].value_counts().index.tolist()
    if cats:
        hboxplot(axes[1], [sized.loc[sized["category"] == c, "size_bytes"] / 1024 for c in cats])
        axes[1].set_yticks(range(1, len(cats) + 1), cats)
        axes[1].set_xscale("log")
    axes[1].set_xlabel("File size (KB, log scale)")
    axes[1].set_title("File size by category (non-empty files)")
    fig.tight_layout()
    save_fig(fig, "02_inventory_extensions_and_sizes")

### 2.3 Duplicate filenames and duplicate contents
* **Duplicate filenames** (same name in different folders) are not necessarily errors, for example `spectrum_1.csv` in each sample folder. However, they show that **the filename alone cannot be the primary key**.
* **Duplicate contents** (identical SHA-256) are byte-for-byte copies. They should be ingested once into Bronze, with all original paths recorded.
* Zero-byte files all share the same checksum, so they are reported separately as empty files rather than as duplicates.

In [ ]:
dup_names, dup_contents = pd.DataFrame(), pd.DataFrame()
if require_data("Section 2.3") and len(inventory):
    real = inventory[~inventory["is_system_file"]].copy()
    real["name_key"] = real["file_name"].str.lower()
    dup_names = (real[real.duplicated("name_key", keep=False)]
                 .groupby("name_key").agg(n_copies=("rel_path", "size"), paths=("rel_path", " | ".join),
                                          distinct_contents=("sha256", "nunique"))
                 .sort_values("n_copies", ascending=False).reset_index())
    hashed = real[real["sha256"].notna() & (real["size_bytes"] > 0)]
    dup_contents = (hashed[hashed.duplicated("sha256", keep=False)]
                    .groupby("sha256").agg(n_copies=("rel_path", "size"), size_bytes=("size_bytes", "first"),
                                           paths=("rel_path", " | ".join))
                    .sort_values("n_copies", ascending=False).reset_index())
    save_table(dup_names, "duplicate_filenames")
    save_table(dup_contents, "duplicate_file_contents")
    print(f"Filenames used more than once: {len(dup_names)} name(s)")
    display(dup_names.head(20))
    print(f"Byte-identical file groups: {len(dup_contents)} group(s) covering {int(dup_contents['n_copies'].sum()) if len(dup_contents) else 0} files")
    display(dup_contents.head(20))
    if not HASH_ALL_FILES:
        print("Note: HASH_ALL_FILES=False, so only files sharing a size with another file were hashed.")

---
## 3 · Initial dataset inspection  ⚠️ REQUIRES DATA

This section defines the **readers** that Section 4 reuses, then shows a seeded random sample of files from each category in detail. Nothing assumes a shared schema: every file and sheet is profiled on its own.

**Terms used in the profile tables**
* *numeric / categorical*: the column's actual stored data type.
* *numeric_stored_as_text*: at least 95 % of the values parse as numbers but are stored as text, for example because of a stray unit string or a decimal comma.
* *mixed_numeric_text*: 20–95 % of the values parse as numbers. This is a real type inconsistency.
* *unit hints*: units mentioned in column names (cm⁻¹, %T, µm, nm, …).

In [ ]:
# ---------- low-level text helpers --------------------------------------------------------
_TEXT_BYTES = bytes(range(32, 127)) + b"\n\r\t\f\b\x1a" + bytes(range(128, 256))
NA_TOKENS = {"", "na", "nan", "n/a", "null", "none", "-", "--"}
DELIMITERS = {"tab": "\t", "comma": ",", "semicolon": ";", "pipe": "|", "whitespace": None}


def looks_binary(raw):
    """NUL bytes, or more than 5 % control bytes, mean the content is binary. Such files are never parsed as text."""
    if not raw:
        return False
    if b"\x00" in raw:
        return True
    return len(raw.translate(None, _TEXT_BYTES)) / len(raw) > 0.05


def decode_head(path, nbytes):
    """Read the first `nbytes` and decode them. Returns (lines, encoding) or (None, None) for binary content."""
    with open(path, "rb") as fh:
        raw = fh.read(nbytes + 1)
    truncated = len(raw) > nbytes
    raw = raw[:nbytes]
    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        encoding = "utf-16"
    elif looks_binary(raw):
        return None, None
    else:
        try:
            raw.decode("utf-8-sig")
            encoding = "utf-8-sig"
        except UnicodeDecodeError as exc:
            if truncated and exc.start >= len(raw) - 4:  # multi-byte character cut by the read limit
                encoding = "utf-8-sig"
            else:
                try:
                    raw.decode("cp1252")
                    encoding = "cp1252"
                except UnicodeDecodeError:
                    encoding = "latin-1"
    lines = raw.decode(encoding, errors="replace").splitlines()
    if truncated and lines:
        lines = lines[:-1]  # the last line may be incomplete
    return lines, encoding


def to_float(token, decimal_comma=False):
    t = str(token).strip().strip('"').strip("'")
    if decimal_comma:
        t = t.replace(",", ".")
    try:
        return float(t)
    except ValueError:
        return None


def split_fields(line, delim):
    if delim is None:
        return line.split()
    return [t.strip() for t in line.split(delim)]


def numeric_fraction(tokens, decimal_comma):
    n_num = sum(to_float(t, decimal_comma) is not None for t in tokens)
    n_na = sum(t.strip().strip('"').lower() in NA_TOKENS for t in tokens)
    return n_num, n_na


def is_data_line(tokens, decimal_comma):
    if len(tokens) < 2:
        return False
    n_num, n_na = numeric_fraction(tokens, decimal_comma)
    return n_num >= 2 and (n_num + n_na) >= 0.5 * len(tokens)


def find_numeric_block(lines, delim, decimal_comma, min_run=5):
    """First run of consistent numeric lines -> (start, run_length, n_fields, completeness)."""
    i, n = 0, len(lines)
    while i < n:
        toks = split_fields(lines[i], delim)
        if not is_data_line(toks, decimal_comma):
            i += 1
            continue
        n_fields, j, completeness = len(toks), i, 0.0
        while j < n:
            t = split_fields(lines[j], delim)
            if len(t) != n_fields or not is_data_line(t, decimal_comma):
                break
            completeness += numeric_fraction(t, decimal_comma)[0] / len(t)
            j += 1
        run = j - i
        if run >= min_run or (j == n and run >= 2):
            return i, run, n_fields, completeness / run
        i = j
    return None, 0, 0, 0.0


def looks_like_axis(values, require_wavenumber_range=True, min_points=20):
    """Monotonic (≥98 % of steps in one direction), enough points, and plausibly spaced."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) < min_points or len(np.unique(v)) < 0.9 * len(v):  # heavy repetition = long format, not one axis
        return False
    d = np.diff(v)
    if max((d > 0).mean(), (d < 0).mean()) < 0.98:
        return False
    nz = np.abs(d[d != 0])
    if len(nz) == 0:
        return False
    if require_wavenumber_range:
        lo, hi = WAVENUMBER_DETECT_RANGE
        step = np.median(nz)
        return v.min() >= lo and v.max() <= hi and 0.01 <= step <= 64
    return True


def sniff_text_file(path):
    """Detect encoding, delimiter, decimal mark, preamble and header row of a delimited text file."""
    lines, encoding = decode_head(path, TEXT_SNIFF_BYTES)
    if lines is None:
        return {"status": "binary_content"}
    if not any(l.strip() for l in lines):
        return {"status": "empty_text"}
    best, best_score = None, None
    for order, (dname, delim) in enumerate(DELIMITERS.items()):
        for decimal_comma in (False, True):
            if decimal_comma and delim == ",":
                continue
            start, run, n_fields, completeness = find_numeric_block(lines, delim, decimal_comma)
            if run == 0:
                continue
            score = (run, round(completeness, 3), -order, -int(decimal_comma))
            if best_score is None or score > best_score:
                best_score = score
                best = dict(start=start, run=run, n_fields=n_fields, delimiter_name=dname,
                            delimiter=delim, decimal="," if decimal_comma else ".")
    if best is None:  # no numeric block: an ordinary table (e.g., metadata)
        sample = "\n".join(lines[:50])
        try:
            delim = csv.Sniffer().sniff(sample, delimiters=",\t;|").delimiter
        except csv.Error:
            delim = ","
        header_row = next(i for i, l in enumerate(lines) if l.strip())
        return {"status": "text_table", "encoding": encoding, "delimiter": delim,
                "delimiter_name": {v: k for k, v in DELIMITERS.items()}.get(delim, delim),
                "decimal": ".", "header_row": header_row, "data_start": header_row + 1,
                "preamble": lines[:header_row], "header_tokens": None}

    start, delim, dc = best["start"], best["delimiter"], best["decimal"] == ","
    header_row, data_start = None, start
    first = split_fields(lines[start], delim)
    first_vals = [to_float(t, dc) for t in first]
    # Wide-by-rows header: the first numeric-looking line is itself a wavenumber sequence.
    numeric_first = [v for v in first_vals if v is not None]
    if looks_like_axis(numeric_first) and start + 1 < len(lines):
        nxt = [to_float(t, dc) for t in split_fields(lines[start + 1], delim)]
        if not looks_like_axis([v for v in nxt if v is not None]):
            header_row, data_start = start, start + 1
    if header_row is None and start > 0:
        prev = max((i for i in range(start) if lines[i].strip()), default=None)
        if prev is not None:
            toks = split_fields(lines[prev], delim)
            n_num, _ = numeric_fraction(toks, dc)
            if len(toks) == best["n_fields"] and n_num < len(toks) / 2:
                header_row = prev
    header_tokens = split_fields(lines[header_row], delim) if header_row is not None else None
    preamble_end = header_row if header_row is not None else start
    return {"status": "numeric_table", "encoding": encoding, **best, "header_row": header_row,
            "data_start": data_start, "preamble": [l for l in lines[:preamble_end] if l.strip()],
            "header_tokens": header_tokens,
            "duplicate_header_tokens": sorted({t for t in header_tokens if header_tokens.count(t) > 1}) if header_tokens else []}


def read_delimited(path, layout, nrows=None):
    """Read with pandas using the sniffed layout. Malformed lines are counted (and logged), not silently skipped."""
    bad_lines = []

    def on_bad(fields):
        bad_lines.append(len(fields))
        return None

    sep = layout["delimiter"]
    kwargs = dict(sep=r"\s+" if sep is None else sep, encoding=layout["encoding"], decimal=layout["decimal"],
                  engine="python", on_bad_lines=on_bad, nrows=nrows, skip_blank_lines=True)
    if layout["header_row"] is not None:
        df = pd.read_csv(path, skiprows=layout["header_row"], header=0, **kwargs)
    else:
        df = pd.read_csv(path, skiprows=layout["data_start"], header=None, **kwargs)
    df.columns = [str(c) for c in df.columns]
    return df, len(bad_lines)


KV_RE = re.compile(r"^\s*#*\s*([A-Za-z][\w .()/%\-]{0,60}?)\s*[:=]\s*(.+?)\s*$")


def parse_preamble(lines, delim):
    """Turn instrument header lines into a {key: value} dict (key: value, key=value, ##KEY=value, key<delim>value)."""
    meta = {}
    for line in lines:
        m = KV_RE.match(line)
        if m:
            key, val = m.groups()
        else:
            toks = [t for t in (split_fields(line, delim) if delim else line.split(None, 1)) if t]
            if len(toks) != 2 or to_float(toks[0]) is not None:
                continue
            key, val = toks
        meta.setdefault(key.strip(), val.strip())
    return meta


# ---------- spreadsheets ------------------------------------------------------------------
def make_unique(names):
    seen, out = Counter(), []
    for n in names:
        seen[n] += 1
        out.append(n if seen[n] == 1 else f"{n}__dup{seen[n]}")
    return out


def promote_header(raw):
    """Find the header row in a sheet read with header=None (title rows above tables are common)."""
    raw = raw.dropna(how="all").dropna(axis=1, how="all")
    if raw.empty:
        return raw, None, None
    head = raw.head(30)
    counts = head.notna().sum(axis=1)
    target = counts.max()
    header_idx = None
    for idx in head.index:
        if counts[idx] < 0.8 * target:
            continue
        vals = head.loc[idx].dropna().tolist()
        n_text = sum(isinstance(v, str) and to_float(v) is None for v in vals)
        numeric_vals = [to_float(v) for v in vals if to_float(v) is not None]
        if n_text >= 0.5 * len(vals) or looks_like_axis(numeric_vals):
            header_idx = idx
        break
    if header_idx is None:
        df = raw.copy()
        df.columns = [f"col_{i}" for i in range(df.shape[1])]
        return df, None, None
    header_values = raw.loc[header_idx].tolist()
    names = [str(v).strip() if pd.notna(v) else f"unnamed_{i}" for i, v in enumerate(header_values)]
    df = raw.loc[raw.index > header_idx].copy()
    df.columns = make_unique(names)
    return df, header_idx, header_values


def read_excel_sheets(path, nrows=None):
    if path.suffix.lower() == ".xls" and ensure_package("xlrd") is None:
        raise ReaderUnavailable("xlrd (needed for legacy .xls)")
    sheets = {}
    with pd.ExcelFile(path) as xl:
        for sheet in xl.sheet_names:
            raw = pd.read_excel(xl, sheet_name=sheet, header=None, nrows=nrows)
            df, header_idx, header_values = promote_header(raw)
            sheets[sheet] = (df, header_idx, header_values)
    return sheets

In [ ]:
# ---------- table profiling ---------------------------------------------------------------
UNIT_HINT_RE = re.compile(r"cm\s*[-−⁻]?\s*1|cm\^-1|1/cm|%\s*t\b|%\s*r\b|\babs\b|absorb|transm|reflect|a\.u\.|kubelka|"
                          r"log\s*\(?1/r|µm|μm|\bum\b|micron|\bnm\b|\bmm\b|\bmg\b|g/l|°c|deg", re.I)
DATE_VALUE_RE = re.compile(r"^\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}")


def column_kind(s):
    if s.notna().sum() == 0:
        return "empty"
    if pd.api.types.is_bool_dtype(s):
        return "boolean"
    if pd.api.types.is_numeric_dtype(s):
        return "numeric"
    if pd.api.types.is_datetime64_any_dtype(s):
        return "datetime"
    txt = s.dropna().astype(str).str.strip()
    frac = pd.to_numeric(txt, errors="coerce").notna().mean()
    frac_dc = pd.to_numeric(txt.str.replace(",", ".", regex=False), errors="coerce").notna().mean()
    if frac >= 0.95:
        return "numeric_stored_as_text"
    if frac_dc >= 0.95:
        return "numeric_with_decimal_comma"
    if 0.2 <= max(frac, frac_dc) < 0.95:
        return "mixed_numeric_text"
    if txt.str.match(DATE_VALUE_RE).mean() >= 0.8:
        return "date_like_text"
    return "categorical"


def profile_table(df, name):
    """Per-column profile and a one-row table summary. No assumption of a shared schema."""
    cols = []
    for c in df.columns:
        s = df[c]
        uniques = s.dropna().unique()
        cols.append({"table": name, "column": c, "dtype": str(s.dtype), "kind": column_kind(s),
                     "n_missing": int(s.isna().sum()), "pct_missing": round(100 * s.isna().mean(), 2) if len(s) else 0,
                     "n_unique": len(uniques), "examples": "; ".join(map(str, uniques[:3]))[:120],
                     "unit_hint": ", ".join(sorted(set(m.group(0) for m in UNIT_HINT_RE.finditer(str(c))))) or None})
    cols = pd.DataFrame(cols)
    kinds = cols["kind"].value_counts() if len(cols) else pd.Series(dtype=int)
    try:
        n_dup = int(df.duplicated().sum())
    except TypeError:  # unhashable cells
        n_dup = None
    summary = {"table": name, "n_rows": len(df), "n_cols": df.shape[1],
               "n_numeric_cols": int(kinds.get("numeric", 0)),
               "n_categorical_cols": int(kinds.get("categorical", 0)),
               "n_inconsistent_type_cols": int(sum(kinds.get(k, 0) for k in
                                                   ("numeric_stored_as_text", "numeric_with_decimal_comma", "mixed_numeric_text"))),
               "pct_missing_cells": round(100 * df.isna().mean().mean(), 2) if df.size else 0,
               "n_duplicate_rows": n_dup}
    return summary, cols

In [ ]:
# ---------- vendor / standard spectral readers ---------------------------------------------
def read_jcamp(path):
    jc = ensure_package("jcamp")
    if jc is None:
        raise ReaderUnavailable("jcamp")
    reader = getattr(jc, "jcamp_readfile", None) or getattr(jc, "readfile")
    data = reader(str(path))
    out = []
    for block in (data.get("children") or [data]):
        if "x" not in block or "y" not in block:
            continue
        meta = {k: v for k, v in block.items() if k not in ("x", "y", "children") and isinstance(v, (str, int, float))}
        out.append(dict(x=block["x"], y=block["y"], x_label=block.get("xunits", "x"), y_label=block.get("yunits"),
                        meta=meta, layout="jcamp_dx", reader="jcamp"))
    return out


def read_spc(path):
    head = read_magic(path, 2)
    if len(head) < 2 or head[1] not in (0x4B, 0x4C, 0x4D):
        raise ValueError(f"SPC version byte {head[1:2].hex() or 'missing'} not recognised (expected 4b, 4c or 4d)")
    mod = ensure_package("spc_io", "spc-io")
    if mod is None:
        raise ReaderUnavailable("spc-io")
    with open(path, "rb") as fh:
        spc = mod.SPC.from_bytes_io(fh)
    meta = {"spc_xtype": str(spc.xtype), "spc_ytype": str(spc.ytype), "spc_date": str(spc.date)}
    return [dict(x=np.asarray(sub.xarray, float), y=np.asarray(sub.yarray, float), x_label=str(spc.xtype),
                 y_label=str(spc.ytype), meta=meta, layout="spc", reader="spc-io") for sub in spc]


def read_opus(path):
    mod = ensure_package("brukeropusreader")
    if mod is None:
        raise ReaderUnavailable("brukeropusreader")
    data = mod.read_file(str(path))
    if "AB" not in data or "AB Data Parameter" not in data:
        raise ValueError(f"No 'AB' spectrum block; blocks present: {sorted(map(str, data.keys()))[:15]}")
    x = np.asarray(data.get_range("AB"), float)
    y = np.asarray(data["AB"], float)[:len(x)]
    meta = {}
    for block in ("AB Data Parameter", "Acquisition", "Optik", "Instrument", "Sample", "Fourier Transformation"):
        for k, v in (data.get(block) or {}).items():
            if isinstance(v, (str, int, float)):
                meta[f"{block}.{k}"] = v
    meta["opus_blocks"] = ", ".join(sorted(map(str, data.keys())))
    return [dict(x=x, y=y, x_label="wavenumber (cm-1, OPUS AB block)",
                 y_label=f"AB block ({meta.get('AB Data Parameter.DPF', 'Y unit not stored')})",
                 meta=meta, layout="opus_ab", reader="brukeropusreader")]


def read_omnic(path):
    if not ENABLE_SPA_READER:
        raise ReaderNotEnabled("Set ENABLE_SPA_READER=True to read OMNIC .spa/.spg with SpectroChemPy, or export CSV from OMNIC.")
    scp = ensure_package("spectrochempy")
    if scp is None:
        raise ReaderUnavailable("spectrochempy")
    ds = scp.read_omnic(str(path))
    x = np.asarray(ds.x.data, float)
    Y = np.atleast_2d(np.asarray(ds.data, float))
    meta = {"y_units": str(getattr(ds, "units", None)), "x_units": str(getattr(ds.x, "units", None))}
    return [dict(x=x, y=Y[i], x_label=f"x ({meta['x_units']})", y_label=meta["y_units"], meta=meta,
                 layout="omnic", reader="spectrochempy") for i in range(Y.shape[0])]


BINARY_READERS = {"jcamp_dx": read_jcamp, "spc_binary": read_spc, "bruker_opus": read_opus, "omnic_binary": read_omnic}

In [ ]:
def load_tables(row):
    """Load a delimited or spreadsheet file into [(table_name, df, context)] for inspection."""
    path = DATA_ROOT / row.rel_path
    if row.category == "delimited_text":
        layout = sniff_text_file(path)
        if layout["status"] in ("binary_content", "empty_text"):
            return [], layout
        df, n_bad = read_delimited(path, layout)
        layout["bad_lines"] = n_bad
        return [(row.rel_path, df, {"header_values": layout.get("header_tokens"), "decimal": layout["decimal"],
                                    "preamble": layout.get("preamble", []), "delimiter": layout["delimiter"]})], layout
    if row.category == "spreadsheet":
        out = []
        for sheet, (df, header_idx, header_values) in read_excel_sheets(path).items():
            out.append((f"{row.rel_path}::{sheet}", df, {"header_values": header_values, "decimal": ".",
                                                        "preamble": [], "delimiter": None, "sheet": sheet,
                                                        "header_row": header_idx}))
        return out, {"status": "spreadsheet", "sheets": len(out)}
    return [], {"status": "not_tabular"}


inspection_summaries, inspection_columns = [], []
if require_data("Section 3 inspection") and len(inventory):
    candidates = inventory[~inventory["is_system_file"] & (inventory["size_bytes"] > 0)]
    for category, group in candidates.groupby("category"):
        group = group.sort_values("rel_path")
        k = min(SAMPLE_FILES_PER_CATEGORY, len(group))
        picks = group.iloc[np.sort(seeded_rng().choice(len(group), size=k, replace=False))]
        display(Markdown(f"### Category `{category}`: {len(group)} file(s); showing {k}"))
        for row in picks.itertuples(index=False):
            display(Markdown(f"**`{row.rel_path}`** ({human_size(row.size_bytes)})"))
            try:
                if category in ("delimited_text", "spreadsheet"):
                    tables, layout = load_tables(row)
                    if category == "delimited_text":
                        print("Detected layout:", {k: v for k, v in layout.items() if k not in ("preamble", "header_tokens")})
                        if layout.get("preamble"):
                            print("Preamble (first 8 lines):", *layout["preamble"][:8], sep="\n   ")
                    for name, df, _ctx in tables:
                        summary, cols = profile_table(df, name)
                        inspection_summaries.append(summary)
                        inspection_columns.append(cols)
                        print(f"Dimensions: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
                        print("Columns:", list(df.columns[:25]), "..." if df.shape[1] > 25 else "")
                        display(df.iloc[:5, :12])
                        display(cols.drop(columns="table").head(25))
                elif category in BINARY_READERS:
                    blocks = BINARY_READERS[category](DATA_ROOT / row.rel_path)
                    print(f"Reader returned {len(blocks)} spectrum block(s).")
                    for b in blocks[:3]:
                        x = np.asarray(b["x"], float)
                        print(f"  {len(x)} points, x {np.nanmin(x):.2f}..{np.nanmax(x):.2f}, x label '{b['x_label']}', y label '{b['y_label']}'")
                        print("  metadata keys:", list(b["meta"])[:20])
                else:
                    print("Not a tabular or spectral format; listed in the inventory only.")
            except (ReaderUnavailable, ReaderNotEnabled) as exc:
                print(f"Reader not available: {exc}")
                logger.warning(f"{row.rel_path}: reader not available: {exc}")
            except Exception as exc:
                print(f"Could not read: {type(exc).__name__}: {exc}")
                logger.error(f"Inspection failed for {row.rel_path}: {type(exc).__name__}: {exc}")
    if inspection_summaries:
        save_table(pd.DataFrame(inspection_summaries), "inspection_table_summaries")
        save_table(pd.concat(inspection_columns, ignore_index=True), "inspection_column_profiles")
        display(Markdown("### Sampled tables: summary"))
        display(pd.DataFrame(inspection_summaries))

---
## 4 · FTIR spectral data identification  ⚠️ REQUIRES DATA

Every candidate file is read **one at a time**. The code then decides whether the file holds spectra and in which layout:

| Layout | Recognised when |
|---|---|
| `wide_rows` | ≥ 20 column headers are numbers forming a monotonic wavenumber sequence (one row = one spectrum) |
| `xy_two_column` / `wide_columns` | One column is a monotonic wavenumber axis; each other numeric column is one spectrum |
| `long` | A wavenumber column repeats for each value of an ID column, and is monotonic within each group |
| `jcamp_dx`, `spc`, `opus_ab`, `omnic` | Read by the vendor or standard reader |

**What is a "wavenumber axis"?** A column that is monotonic (≥ 98 % of steps go in one direction), has at least 20 points, has values in `WAVENUMBER_DETECT_RANGE`, and has a point spacing between 0.01 and 64 cm⁻¹. A header such as `Wavenumber`, `cm-1`, or `1/cm` is checked first. A *wavelength* column (nm/µm) is recorded as wavelength and **not converted**.

**What the signal axis represents.** The y-axis meaning is recorded from two separate sources:
* **declared**: taken from a column header, the JCAMP `YUNITS` field, the SPC `ytype` field, or file context (sheet name, preamble, filename).
* **heuristic**: inferred from the values. A 0–100 scale suggests %T or %R. In transmittance spectra, peaks point *down* (most points sit near the top baseline). In absorbance spectra, peaks point *up*. **This is only a hint**, so **transmittance is never converted to absorbance here**.

**Point spacing vs. spectral resolution.** The spacing between points is *not* the instrument's optical resolution. Zero-filling during the Fourier transform often produces spacing that is 2× or 4× finer than the nominal resolution. The nominal resolution has to come from metadata, for example a preamble `Resolution` field or the OPUS `Acquisition.RES` parameter.

In [ ]:
WN_NAME_RE = re.compile(r"wave\s*_?\s*num|\bwn\b|cm\s*[-−⁻]?\s*1|cm\^-1|1/cm", re.I)
WL_NAME_RE = re.compile(r"wave\s*_?\s*length|\bnm\b|\(nm\)|µm|μm|\bum\b|micron", re.I)
Y_KIND_PATTERNS = [
    ("transmittance", re.compile(r"transm|%\s*t\b|\bt\s*%", re.I)),
    ("reflectance", re.compile(r"reflect|%\s*r\b|kubelka|\bk-?m\b|log\s*\(?\s*1\s*/\s*r", re.I)),
    ("absorbance", re.compile(r"absorb|\babs\b|\bab block\b", re.I)),
    ("intensity/arbitrary", re.compile(r"intens|counts|signal|single\s*beam|a\.u\.|arbitr", re.I)),
]
POLYMER_COL_RE = re.compile(r"polymer|material|plastic|resin|composition|identif|\bclass\b|^type$", re.I)


def declared_y_kind(text):
    if not text:
        return None
    for kind, pattern in Y_KIND_PATTERNS:
        if pattern.search(str(text)):
            return kind
    return None


def declared_x_unit(label):
    if label and WN_NAME_RE.search(str(label)):
        return "cm-1"
    if label and WL_NAME_RE.search(str(label)):
        return "wavelength (not wavenumber)"
    return None


def numeric_view(df, decimal_comma=False):
    """Numeric copy of columns that are (or are ≥95 % parseable as) numbers. The original df is not modified."""
    out = {}
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if pd.api.types.is_numeric_dtype(s):
            out[c] = s.astype(float)
            continue
        txt = s.astype(str).str.strip()
        if decimal_comma:
            txt = txt.str.replace(",", ".", regex=False)
        conv = pd.to_numeric(txt, errors="coerce")
        if s.notna().sum() and conv.notna().sum() >= 0.95 * s.notna().sum():
            out[c] = conv.astype(float)
    return pd.DataFrame(out, index=df.index)


def is_row_index(v):
    v = np.asarray(v, float)
    return len(v) > 2 and (np.array_equal(v, np.arange(len(v))) or np.array_equal(v, np.arange(1, len(v) + 1)))


def first_text_value(df, i, columns):
    for c in columns:
        v = df.iloc[i][c]
        if pd.notna(v) and str(v).strip():
            return str(v).strip()
    return None


def spectra_from_table(df, header_values=None, decimal_comma=False, context=""):
    """Return (list of raw spectrum dicts, layout name). Never modifies df."""
    if df.empty:
        return [], "empty_table"
    blocks = []
    hv = header_values if header_values is not None else list(df.columns)
    hv_float = [to_float(h, decimal_comma) if h is not None and not (isinstance(h, float) and np.isnan(h)) else None
                for h in hv]
    wn_cols = [c for c, f in zip(df.columns, hv_float) if f is not None]
    # (a) wide_rows
    if len(wn_cols) >= 20:
        xs = np.array([f for f in hv_float if f is not None])
        if looks_like_axis(xs):
            label_cols = [c for c in df.columns if c not in wn_cols]
            polymer_cols = [c for c in label_cols if POLYMER_COL_RE.search(str(c))]
            Y = df[wn_cols].apply(lambda s: pd.to_numeric(s, errors="coerce")).to_numpy(float)
            for i in range(len(df)):
                meta = {str(c): df.iloc[i][c] for c in label_cols}
                blocks.append(dict(x=xs, y=Y[i], x_label="column headers (wavenumber)", y_label=None,
                                   label=first_text_value(df, i, [c for c in label_cols if c not in polymer_cols]),
                                   polymer_label_in_file=first_text_value(df, i, polymer_cols),
                                   meta=meta, layout="wide_rows"))
            return blocks, "wide_rows"
    num = numeric_view(df, decimal_comma)
    named = [c for c in num.columns if WN_NAME_RE.search(str(c))]
    wl_named = [c for c in num.columns if WL_NAME_RE.search(str(c)) and c not in named]
    x_col, x_unit = None, None
    for c in named + [c for c in num.columns if c not in named and c not in wl_named]:
        if looks_like_axis(num[c].to_numpy()):
            x_col, x_unit = c, "cm-1" if c in named else "undeclared (values consistent with cm-1)"
            break
    if x_col is None:
        for c in wl_named:
            if looks_like_axis(num[c].to_numpy(), require_wavenumber_range=False):
                x_col, x_unit = c, "wavelength (not wavenumber)"
                break
    if x_col is None:
        return try_long_format(df, num, named, context)
    y_cols = [c for c in num.columns if c != x_col and not is_row_index(num[c].to_numpy())]
    if not y_cols:
        return [], "axis_without_values"
    layout = "xy_two_column" if len(y_cols) == 1 else "wide_columns"
    file_meta = {str(c): df[c].dropna().iloc[0] for c in df.columns
                 if c not in num.columns and df[c].nunique(dropna=True) == 1}
    for c in y_cols:
        blocks.append(dict(x=num[x_col].to_numpy(), y=num[c].to_numpy(), x_label=x_col, y_label=c,
                           x_unit=x_unit, label=None if layout == "xy_two_column" else str(c),
                           polymer_label_in_file=None, meta=dict(file_meta), layout=layout))
    return blocks, layout


def try_long_format(df, num, named_x, context=""):
    cand_x = named_x or [c for c in num.columns if num[c].nunique() < 0.9 * len(num)
                         and num[c].between(*WAVENUMBER_DETECT_RANGE).mean() > 0.95]
    id_cols = [c for c in df.columns if c not in cand_x and 2 <= df[c].nunique() <= max(2, len(df) / 10)]
    for xc in cand_x:
        for ic in id_cols:
            groups = df.groupby(ic, sort=False).indices
            xv = num[xc].to_numpy()
            if sum(looks_like_axis(xv[pos]) for pos in groups.values()) < 0.9 * len(groups):
                continue
            value_cols = [c for c in num.columns if c not in (xc, ic)]
            if not value_cols:
                continue
            polymer_cols = [c for c in df.columns if c not in num.columns and c != ic and POLYMER_COL_RE.search(str(c))]
            blocks = []
            for gid, pos in groups.items():
                poly = None
                if polymer_cols:
                    vals = df.iloc[pos][polymer_cols[0]].dropna().unique()
                    poly = str(vals[0]) if len(vals) == 1 else None
                for vc in value_cols:
                    blocks.append(dict(x=xv[pos], y=num[vc].to_numpy()[pos], x_label=xc, y_label=vc, x_unit="cm-1",
                                       label=str(gid), polymer_label_in_file=poly, meta={str(ic): gid}, layout="long"))
            return blocks, "long"
    return [], "not_spectral"

In [ ]:
QC_THRESHOLDS = {
    "spacing_rel_spread": 0.01,     # (max-min)/median point spacing above this = non-uniform grid
    "absorbance_high": 3.0,         # A > 3 means < 0.1 % of light transmitted: detector non-linearity is likely
    "absorbance_negative": -0.05,   # clearly negative absorbance suggests a baseline or background problem
    "percent_high": 100.5,          # %T or %R above 100 is non-physical (background mismatch?)
    "percent_low": -0.5,
    "edge_drift_frac": 0.25,        # |end-to-end baseline drift| as a fraction of the signal range
    "low_snr": 20.0,                # signal range / robust noise estimate
    "robust_z_extreme": 5.0,        # spectrum's 99th percentile vs. the dataset (median/MAD)
}


def classify_y_values(p01, p50, p99):
    """Heuristic only. Returns (value_scale, peak_orientation, heuristic_kind)."""
    dyn = p99 - p01
    if not np.isfinite(dyn) or dyn <= 0:
        return "flat", "undetermined", "undetermined"
    pos = (p50 - p01) / dyn
    orientation = "peaks_down" if pos > 0.6 else "peaks_up" if pos < 0.4 else "undetermined"
    if p99 > 1.5 and p01 >= -5 and p99 <= 120:
        scale = "0-100 (percent-like)"
    elif p01 >= -0.5 and p99 <= 1.2 and p50 > 0.4:
        scale = "0-1 (fraction-like)"
    elif p01 >= -0.5 and p99 <= 4:
        scale = "0-4 (absorbance-like)"
    else:
        scale = "arbitrary"
    if orientation == "peaks_down" and scale.startswith(("0-100", "0-1")):
        kind = "T or R (heuristic)"
    elif orientation == "peaks_up" and scale in ("0-4 (absorbance-like)", "0-1 (fraction-like)"):
        kind = "absorbance (heuristic)"
    else:
        kind = "undetermined"
    return scale, orientation, kind


def spectrum_metrics(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = {"n_points": len(x), "n_nonfinite_x": int((~np.isfinite(x)).sum()), "n_nonfinite_y": int((~np.isfinite(y)).sum())}
    ok = np.isfinite(x) & np.isfinite(y)
    xv, yv = x[ok], y[ok]
    m["n_valid_points"] = int(ok.sum())
    if len(xv) < 3:
        m["ordering"] = "insufficient_points"
        return m
    d = np.diff(xv)
    m["ordering"] = "ascending" if np.all(d > 0) else "descending" if np.all(d < 0) else "non_monotonic"
    signs = np.sign(d[d != 0])
    m["n_order_reversals"] = int(np.sum(signs[1:] != signs[:-1]))
    uniq, counts = np.unique(xv, return_counts=True)
    m["n_duplicate_x"] = int((counts > 1).sum())
    if m["n_duplicate_x"]:
        spread = pd.Series(yv).groupby(xv).agg(lambda g: g.max() - g.min())
        m["n_duplicate_x_conflicting_y"] = int((spread > 1e-9).sum())
    else:
        m["n_duplicate_x_conflicting_y"] = 0
    nz = np.abs(d[d != 0])
    step = float(np.median(nz)) if len(nz) else np.nan
    m.update(x_min=float(xv.min()), x_max=float(xv.max()), median_spacing=step,
             min_spacing=float(nz.min()) if len(nz) else np.nan, max_spacing=float(nz.max()) if len(nz) else np.nan)
    m["spacing_rel_spread"] = (m["max_spacing"] - m["min_spacing"]) / step if step else np.nan
    p = np.percentile(yv, [0, 1, 5, 50, 95, 99, 100])
    m.update(y_min=p[0], y_p01=p[1], y_p05=p[2], y_median=p[3], y_p95=p[4], y_p99=p[5], y_max=p[6])
    dyn = p[5] - p[1]
    ys = yv[np.argsort(xv, kind="stable")]
    d2 = np.diff(ys, 2)
    # Robust white-noise sigma from second differences: Var(Δ²y) = 6σ² for white noise.
    noise = 1.4826 * np.median(np.abs(d2 - np.median(d2))) / np.sqrt(6) if len(d2) else np.nan
    m["noise_sigma_est"] = float(noise)
    m["snr_est"] = float(dyn / noise) if noise and noise > 0 else np.nan
    k = max(3, len(ys) // 20)
    m["edge_drift_frac"] = float((np.median(ys[-k:]) - np.median(ys[:k])) / dyn) if dyn > 0 else 0.0
    m["value_scale"], m["peak_orientation"], m["y_kind_heuristic"] = classify_y_values(p[1], p[3], p[5])
    m["xy_fingerprint"] = hashlib.sha1(np.round(xv, 4).tobytes() + np.round(yv, 6).tobytes()).hexdigest()
    return m

In [ ]:
SKIP_STATUS = {"image": "not_applicable", "document": "not_applicable", "archive": "not_extracted",
               "structured_text": "not_parsed_in_eda", "columnar": "not_parsed_in_eda",
               "unknown": "unknown_format", "no_extension": "unknown_format"}


def process_file(row):
    """Read one file. Returns (file_info, spectrum_blocks, tables). Every outcome has an explicit status."""
    info = {"rel_path": row.rel_path, "category": row.category, "status": None, "reader": None, "layout": None,
            "n_spectra": 0, "n_tables": 0, "bad_lines": 0, "encoding": None, "delimiter": None, "decimal": None,
            "duplicate_header_tokens": None, "message": ""}
    path = DATA_ROOT / row.rel_path
    if row.size_bytes == 0:
        info["status"] = "empty_file"
        return info, [], []
    if row.category not in SPECTRAL_CANDIDATE_CATEGORIES:
        info["status"] = SKIP_STATUS.get(row.category, "not_parsed_in_eda")
        return info, [], []
    if row.category == "delimited_text" and row.size_bytes > MAX_TEXT_FILE_MB * 1e6:
        info["status"] = "skipped_too_large"
        info["message"] = f"> {MAX_TEXT_FILE_MB} MB; raise MAX_TEXT_FILE_MB to include"
        return info, [], []
    blocks, tables = [], []
    try:
        if row.category in ("delimited_text", "spreadsheet"):
            loaded, layout = load_tables(row)
            if layout["status"] == "binary_content":
                info["status"] = "unexpected_binary_content"
                info["message"] = "text extension but binary bytes; not parsed as text"
                return info, [], []
            if layout["status"] == "empty_text":
                info["status"] = "empty_file"
                info["message"] = "only whitespace"
                return info, [], []
            info.update(reader="pandas", bad_lines=layout.get("bad_lines", 0), encoding=layout.get("encoding"),
                        delimiter=layout.get("delimiter_name"), decimal=layout.get("decimal"),
                        duplicate_header_tokens="; ".join(layout.get("duplicate_header_tokens") or []) or None)
            layouts = []
            for name, df, ctx in loaded:
                file_meta = parse_preamble(ctx["preamble"], ctx["delimiter"])
                context = " ".join([row.file_name, ctx.get("sheet") or ""] + ctx["preamble"][:30])
                found, lay = spectra_from_table(df, ctx["header_values"], ctx["decimal"] == ",", context)
                layouts.append(lay)
                for b in found:
                    b["meta"] = {**file_meta, **b["meta"]}
                    if ctx.get("sheet"):
                        b["meta"]["sheet"] = ctx["sheet"]
                    b["context"] = context
                blocks.extend(found)
                if not found:
                    tables.append((name, df, file_meta))
            info["layout"] = ", ".join(sorted(set(layouts)))
        else:
            blocks = BINARY_READERS[row.category](path)
            info["reader"] = blocks[0]["reader"] if blocks else None
            info["layout"] = blocks[0]["layout"] if blocks else None
            for b in blocks:
                b["context"] = row.file_name
        info["n_spectra"], info["n_tables"] = len(blocks), len(tables)
        info["status"] = "spectral" if blocks else ("tabular_non_spectral" if tables else "no_content_found")
    except ReaderUnavailable as exc:
        info["status"], info["message"] = "reader_unavailable", str(exc)
        logger.warning(f"{row.rel_path}: reader unavailable: {exc}")
    except ReaderNotEnabled as exc:
        info["status"], info["message"] = "reader_not_enabled", str(exc)
        logger.info(f"{row.rel_path}: {exc}")
    except Exception as exc:
        info["status"], info["message"] = "parse_error", f"{type(exc).__name__}: {exc}"[:500]
        logger.error(f"Parse error in {row.rel_path}: {type(exc).__name__}: {exc}")
    if info["bad_lines"]:
        logger.warning(f"{row.rel_path}: {info['bad_lines']} malformed line(s) had the wrong number of fields")
    return info, blocks, tables

In [ ]:
parse_log, spectra_df, spectrum_meta_long = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
plot_sample, metadata_tables = [], {}

if require_data("Section 4 spectral identification") and len(inventory):
    todo = inventory[~inventory["is_system_file"]].sort_values("rel_path")
    if MAX_FILES_TO_PARSE and len(todo) > MAX_FILES_TO_PARSE:
        todo = todo.sample(n=MAX_FILES_TO_PARSE, random_state=RANDOM_SEED).sort_values("rel_path")
        print(f"MAX_FILES_TO_PARSE={MAX_FILES_TO_PARSE}: parsing a seeded random subset.")
    reservoir_rng = seeded_rng()
    parse_rows, spectrum_rows, meta_rows, n_seen = [], [], [], 0
    for row in tqdm(list(todo.itertuples(index=False)), desc="Reading files"):
        info, blocks, tables = process_file(row)
        parse_rows.append(info)
        src_key = row.sha256 if isinstance(row.sha256, str) else hashlib.sha1(row.rel_path.encode()).hexdigest()
        for i, b in enumerate(blocks):
            sid = f"{src_key[:16]}-{i:05d}"
            m = spectrum_metrics(b["x"], b["y"])
            y_decl = declared_y_kind(b.get("y_label"))
            y_src = "column/reader label" if y_decl else None
            if not y_decl:
                y_decl = declared_y_kind(b.get("context"))
                y_src = "file context (name/sheet/preamble)" if y_decl else None
            rec = {"spectrum_id": sid, "rel_path": row.rel_path, "index_in_file": i, "layout": b["layout"],
                   "x_label": str(b.get("x_label")), "y_label": None if b.get("y_label") is None else str(b["y_label"]),
                   "x_unit_declared": b.get("x_unit") or declared_x_unit(b.get("x_label")),
                   "y_kind_declared": y_decl, "y_kind_source": y_src, "label": b.get("label"),
                   "polymer_label_in_file": b.get("polymer_label_in_file"), **m}
            spectrum_rows.append(rec)
            for k, v in (b.get("meta") or {}).items():
                meta_rows.append({"spectrum_id": sid, "rel_path": row.rel_path, "key": str(k), "value": str(v)[:200]})
            compact = {"spectrum_id": sid, "rel_path": row.rel_path, "x": np.asarray(b["x"], float),
                       "y": np.asarray(b["y"], float), "label": b.get("label"),
                       "polymer_label_in_file": b.get("polymer_label_in_file"),
                       "y_label": rec["y_label"], "y_kind": y_decl or m.get("y_kind_heuristic"),
                       "x_unit": rec["x_unit_declared"]}
            n_seen += 1  # reservoir sampling keeps a uniform random sample of fixed size
            if len(plot_sample) < MAX_SPECTRA_FOR_PLOTS:
                plot_sample.append(compact)
            else:
                j = reservoir_rng.integers(0, n_seen)
                if j < MAX_SPECTRA_FOR_PLOTS:
                    plot_sample[j] = compact
        for name, df, file_meta in tables:
            if len(df) <= MAX_METADATA_TABLE_ROWS:
                metadata_tables[name] = df
            else:
                logger.warning(f"{name}: {len(df)} rows exceeds MAX_METADATA_TABLE_ROWS; not kept for metadata analysis")
    parse_log = pd.DataFrame(parse_rows)
    spectra_df = pd.DataFrame(spectrum_rows)
    spectrum_meta_long = pd.DataFrame(meta_rows, columns=["spectrum_id", "rel_path", "key", "value"])
    save_table(parse_log, "parse_log")
    save_table(spectra_df, "spectra_summary")
    save_table(spectrum_meta_long, "spectrum_embedded_metadata_long")

    display(Markdown("### File parse status"))
    display(parse_log.groupby(["category", "status"]).size().rename("n_files").reset_index())
    print(f"Spectra identified: {len(spectra_df):,} from {int((parse_log['status'] == 'spectral').sum()):,} file(s)")
    print(f"Non-spectral tables kept for metadata analysis: {len(metadata_tables)}")
    problems = parse_log[parse_log["status"].isin(["parse_error", "unexpected_binary_content", "reader_unavailable"])]
    if len(problems):
        display(Markdown("**Files that could not be read** (also in `tables/parse_log.csv` and the log):"))
        display(problems[["rel_path", "status", "message"]].head(30))

### 4.1 Spectral axis and signal checks

In [ ]:
if require_data("Section 4.1") and len(spectra_df):
    s = spectra_df
    checks = pd.DataFrame({
        "check": ["spectra", "wavenumber min (cm-1), range across spectra", "wavenumber max (cm-1), range across spectra",
                  "points per spectrum (min / median / max)", "median point spacing (min / median / max)",
                  "ordering", "spectra with non-uniform spacing", "spectra with duplicate wavenumbers",
                  "  ...of which duplicates disagree in y", "spectra with non-finite x or y values",
                  "declared x units", "declared y quantity", "heuristic y quantity", "spectral layouts"],
        "result": [
            f"{len(s):,}",
            f"{s['x_min'].min():.1f} – {s['x_min'].max():.1f}" if "x_min" in s else "n/a",
            f"{s['x_max'].min():.1f} – {s['x_max'].max():.1f}" if "x_max" in s else "n/a",
            f"{s['n_points'].min():,} / {int(s['n_points'].median()):,} / {s['n_points'].max():,}",
            f"{s['median_spacing'].min():.4g} / {s['median_spacing'].median():.4g} / {s['median_spacing'].max():.4g}" if "median_spacing" in s else "n/a",
            dict(s["ordering"].value_counts()),
            int((s.get("spacing_rel_spread", pd.Series(dtype=float)) > QC_THRESHOLDS["spacing_rel_spread"]).sum()),
            int((s.get("n_duplicate_x", pd.Series(dtype=float)) > 0).sum()),
            int((s.get("n_duplicate_x_conflicting_y", pd.Series(dtype=float)) > 0).sum()),
            int(((s["n_nonfinite_x"] > 0) | (s["n_nonfinite_y"] > 0)).sum()),
            dict(s["x_unit_declared"].fillna("undeclared").value_counts()),
            dict(s["y_kind_declared"].fillna("undeclared").value_counts()),
            dict(s["y_kind_heuristic"].fillna("n/a").value_counts()) if "y_kind_heuristic" in s else "n/a",
            dict(s["layout"].value_counts()),
        ]})
    save_table(checks, "spectral_axis_checks")
    display(checks)

    grids = (s.assign(x_min_r=s["x_min"].round(1), x_max_r=s["x_max"].round(1), step_r=s["median_spacing"].round(4))
             .groupby(["x_min_r", "x_max_r", "n_points", "step_r"]).size().rename("n_spectra")
             .sort_values(ascending=False).reset_index())
    save_table(grids, "distinct_wavenumber_grids")
    print(f"Distinct wavenumber grids (min, max, n_points, spacing): {len(grids)}")
    display(grids.head(15))
    mixed_units = s["y_kind_declared"].dropna().nunique() > 1 or s["y_kind_heuristic"].isin(["T or R (heuristic)"]).any() and s["y_kind_heuristic"].isin(["absorbance (heuristic)"]).any()
    print("Signal quantity consistent across spectra:", "NO: mixed or conflicting y quantities (see table)" if mixed_units else "no conflict detected (still verify against lab records)")

---
## 5 · Initial spectral visualizations  ⚠️ REQUIRES DATA

Plotting conventions:
* The **wavenumber axis runs from high to low** (4000 → 400 cm⁻¹), as is standard for FTIR.
* Raw values are plotted unchanged. When spectra with different scales are overlaid, a **display-only** min–max scaling is used and labelled as such. The data is not modified.
* Axis labels use the *declared* quantity when one exists. Otherwise they say "units undeclared".

In [ ]:
def x_axis_label(unit):
    if unit and unit.startswith("wavelength"):
        return "Wavelength (unit as in file)"
    return "Wavenumber (cm$^{-1}$)" + ("" if unit == "cm-1" else " [inferred from values]")


def y_axis_label(spec):
    return spec["y_label"] if spec.get("y_label") else f"Signal (units undeclared; {spec.get('y_kind') or 'unknown'})"


def orient_axis(ax, unit):
    if not (unit and unit.startswith("wavelength")):
        ax.invert_xaxis()  # FTIR convention: high wavenumber on the left


def minmax(y):
    lo, hi = np.nanmin(y), np.nanmax(y)
    return (y - lo) / (hi - lo) if hi > lo else y * 0


if require_data("Section 5") and plot_sample:
    rng = seeded_rng()
    valid = [p for p in plot_sample if np.isfinite(p["y"]).sum() >= 20]
    rep = valid[rng.integers(len(valid))] if valid else None
    if rep is not None:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(rep["x"], rep["y"], lw=0.9, color="#1f4e79")
        orient_axis(ax, rep["x_unit"])
        ax.set_xlabel(x_axis_label(rep["x_unit"]))
        ax.set_ylabel(y_axis_label(rep))
        ax.set_title(f"Raw spectrum (seeded random pick): {rep['rel_path']}" + (f" · {rep['label']}" if rep["label"] else ""), fontsize=9)
        save_fig(fig, "05a_representative_raw_spectrum")

    picks = [valid[i] for i in rng.choice(len(valid), size=min(8, len(valid)), replace=False)] if valid else []
    if picks:
        fig, axes = plt.subplots(2, 1, figsize=(10, 7.5), sharex=True)
        for p in picks:
            name = (p["label"] or Path(p["rel_path"]).stem)[:30]
            axes[0].plot(p["x"], p["y"], lw=0.8, label=name)
            axes[1].plot(p["x"], minmax(p["y"]), lw=0.8)
        axes[0].set_ylabel("Raw signal (as stored)")
        axes[0].set_title(f"Overlay of {len(picks)} randomly selected spectra (raw)")
        axes[1].set_ylabel("Min–max scaled\n(DISPLAY ONLY)")
        axes[1].set_title("Same spectra, scaled to [0, 1] for shape comparison only (data not modified)")
        orient_axis(axes[1], picks[0]["x_unit"])
        axes[1].set_xlabel(x_axis_label(picks[0]["x_unit"]))
        axes[0].legend(fontsize=7, ncol=2, loc="best")
        fig.tight_layout()
        save_fig(fig, "05b_overlay_spectra")

In [ ]:
if require_data("Section 5 distributions") and len(spectra_df) and "x_min" in spectra_df:
    s = spectra_df.dropna(subset=["x_min", "x_max"]).sort_values(["x_min", "x_max"])
    shown = s if len(s) <= 400 else s.sample(400, random_state=RANDOM_SEED).sort_values(["x_min", "x_max"])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    axes[0].hlines(range(len(shown)), shown["x_min"], shown["x_max"], lw=1, color="#4C72B0")
    axes[0].invert_xaxis()
    axes[0].set_xlabel("Wavenumber (cm$^{-1}$)")
    axes[0].set_ylabel("Spectrum (sorted)")
    axes[0].set_title(f"Wavenumber coverage ({len(shown)} spectra)")
    for v, txt in ((4000, "4000"), (400, "400")):
        axes[0].axvline(v, color="grey", ls=":", lw=1)
    axes[1].hist(s["n_points"], bins=min(40, max(5, s["n_points"].nunique())), color="#55A868")
    axes[1].set_xlabel("Points per spectrum")
    axes[1].set_ylabel("Number of spectra")
    axes[1].set_title("Spectral length")
    axes[2].hist(s["median_spacing"].dropna(), bins=40, color="#C44E52")
    axes[2].set_xlabel("Median point spacing (cm$^{-1}$)")
    axes[2].set_title("Point spacing (≠ optical resolution)")
    fig.tight_layout()
    save_fig(fig, "05c_wavenumber_ranges_lengths_spacing")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    kinds = s["y_kind_declared"].fillna("undeclared: " + s["y_kind_heuristic"].fillna("n/a"))
    order = kinds.value_counts().index.tolist()
    hboxplot(axes[0], [s.loc[kinds == k, "y_p99"].dropna() for k in order])
    axes[0].set_yticks(range(1, len(order) + 1), [k[:35] for k in order], fontsize=8)
    axes[0].set_xlabel("99th-percentile signal per spectrum (raw)")
    axes[0].set_title("Signal magnitude by y quantity")
    snr = s["snr_est"].replace([np.inf, -np.inf], np.nan)
    axes[1].scatter(s["noise_sigma_est"], snr, s=8, alpha=0.6)
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].axhline(QC_THRESHOLDS["low_snr"], color="red", ls="--", lw=1, label=f"SNR = {QC_THRESHOLDS['low_snr']:.0f}")
    axes[1].set_xlabel("Robust noise estimate (σ)")
    axes[1].set_ylabel("Estimated SNR")
    axes[1].set_title("Noise vs. signal-to-noise (heuristic)")
    axes[1].legend()
    fig.tight_layout()
    save_fig(fig, "05d_intensity_and_noise_summary")

In [ ]:
def plot_missingness(tables, max_tables=6, max_rows=400):
    """Missing-value matrices: black = missing. Rows are sub-sampled with a fixed seed for large tables."""
    items = list(tables.items())[:max_tables]
    if not items:
        return
    fig, axes = plt.subplots(len(items), 1, figsize=(12, 2.6 * len(items)), squeeze=False)
    for ax, (name, df) in zip(axes[:, 0], items):
        view = df if len(df) <= max_rows else df.sample(max_rows, random_state=RANDOM_SEED).sort_index()
        ax.imshow(view.isna().to_numpy().T, aspect="auto", cmap="Greys", interpolation="nearest", vmin=0, vmax=1)
        ax.set_yticks(range(view.shape[1]), [str(c)[:25] for c in view.columns], fontsize=7)
        ax.set_xlabel("Row")
        ax.set_title(f"{name}: {100 * df.isna().mean().mean():.1f}% cells missing", fontsize=9)
        ax.grid(False)
    fig.tight_layout()
    save_fig(fig, "05e_metadata_missingness")


if require_data("Section 5 missing values"):
    if len(spectra_df):
        frac = (spectra_df["n_nonfinite_y"] + spectra_df["n_nonfinite_x"]) / spectra_df["n_points"].clip(lower=1)
        print(f"Spectra with any non-finite/missing points: {int((frac > 0).sum())} of {len(spectra_df)}")
        if (frac > 0).any():
            fig, ax = plt.subplots(figsize=(8, 3.5))
            ax.hist(100 * frac[frac > 0], bins=30, color="#8172B2")
            ax.set_xlabel("% of points missing or non-finite")
            ax.set_ylabel("Spectra")
            ax.set_title("Missing values within spectra (only spectra with any)")
            save_fig(fig, "05f_spectra_missing_values")
    if metadata_tables:
        plot_missingness(metadata_tables)
    else:
        print("No non-spectral (metadata) tables found to visualise.")

In [ ]:
def plot_label_comparison(samples, labels, title, fname, max_labels=8):
    """Mean ± SD per label on a common grid. Interpolation and scaling are DISPLAY-ONLY and in memory."""
    labelled = [p for p in samples if labels.get(p["spectrum_id"]) and np.isfinite(p["y"]).sum() >= 20]
    if not labelled:
        display(Markdown("> No labelled spectra in the plotting sample; comparison not drawn. **Labels are never invented.**"))
        return
    kind = Counter(p["y_kind"] for p in labelled).most_common(1)[0][0]
    labelled = [p for p in labelled if p["y_kind"] == kind]  # never compare different y quantities
    by_label = defaultdict(list)
    for p in labelled:
        by_label[labels[p["spectrum_id"]]].append(p)
    if len(by_label) < 2:
        display(Markdown(f"> Only one label present (`{next(iter(by_label))}`); a comparison needs at least two."))
        return
    top = sorted(by_label, key=lambda k: -len(by_label[k]))[:max_labels]
    lo = max(np.nanmin(p["x"]) for k in top for p in by_label[k])
    hi = min(np.nanmax(p["x"]) for k in top for p in by_label[k])
    if hi - lo < 200:
        display(Markdown("> Labelled spectra share < 200 cm⁻¹ of common range; comparison not drawn."))
        return
    grid = np.linspace(hi, lo, 1500)
    fig, ax = plt.subplots(figsize=(10, 1.1 * len(top) + 2))
    for off, k in enumerate(top):
        curves = []
        for p in by_label[k]:
            ok = np.isfinite(p["x"]) & np.isfinite(p["y"])
            order = np.argsort(p["x"][ok])
            curves.append(minmax(np.interp(grid, p["x"][ok][order], p["y"][ok][order])))
        curves = np.array(curves)
        mean, sd = curves.mean(0), curves.std(0)
        ax.plot(grid, mean + 1.2 * off, lw=1, label=f"{k} (n={len(curves)})")
        ax.fill_between(grid, mean - sd + 1.2 * off, mean + sd + 1.2 * off, alpha=0.2)
    ax.invert_xaxis()
    ax.set_xlabel("Wavenumber (cm$^{-1}$)")
    ax.set_ylabel("Scaled signal + offset (DISPLAY ONLY)")
    ax.set_yticks([])
    ax.set_title(f"{title}\n(y quantity: {kind}; mean ± SD of min–max scaled spectra; vertical offsets for clarity)", fontsize=9)
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
    save_fig(fig, fname)


if require_data("Section 5 polymer comparison"):
    in_file_labels = {p["spectrum_id"]: p["polymer_label_in_file"] for p in plot_sample if p.get("polymer_label_in_file")}
    print(f"Spectra (in plotting sample) with a polymer label stored in the spectral file itself: {len(in_file_labels)}")
    plot_label_comparison(plot_sample, in_file_labels, "Spectra grouped by polymer label found in spectral files",
                          "05g_polymer_comparison_in_file_labels")
    print("Labels from separate metadata tables are linked in Section 6.3.")

---
## 6 · Metadata exploration  ⚠️ REQUIRES DATA

Metadata can come from three places:
1. **Separate tables** (spreadsheets or CSVs that are not spectra).
2. **Instrument headers inside spectral files** (preamble lines, JCAMP `##` fields, OPUS parameter blocks).
3. **Label columns in wide-format spectral tables.**

Each field is assigned a category from its **name** (a heuristic, so please review the result) and a candidate **PostgreSQL type** from its **values**. Section 6.2 then tests whether spectra can be linked to metadata rows by identifier.

In [ ]:
FIELD_CATEGORIES = [  # order matters: first match wins
    ("sample_preparation", r"prep|digest|h2o2|koh|fenton|density|nacl|zncl|filter|sieve|extract|treat|wash|dry"),
    ("measurement_settings", r"resol|scan|aperture|range|gain|apodi|mode|velocity|zero.?fill|phase|background|accum|\bres\b|\bnss\b"),
    ("instrument", r"instrument|spectrometer|detector|crystal|\batr\b|microscope|model|serial|accessory|beam.?split"),
    ("qc_annotation", r"\bqc\b|blank|control|flag|note|comment|quality|hqi|score|confiden|remark|valid"),
    ("polymer_type", r"polymer|material|plastic|resin|composition|identif|match|\bclass\b|^type$"),
    ("particle_shape", r"shape|morph|fib(er|re)|fragment|film|foam|bead|pellet"),
    ("particle_size", r"size|length|width|diam|area|feret|µm|μm|\bum\b|micron|\bmm\b|perimeter|thick"),
    ("particle_color", r"colou?r"),
    ("sampling_location", r"site|locat|station|\blat|\blon|coord|river|beach|lake|bay|depth|region|county|transect|gps|utm|easting|northing"),
    ("collection_date", r"date|time|collect|sampled|year|month|\bday\b"),
    ("operator_project", r"operator|analyst|user|project|batch|\blab\b"),
    ("sample_identifier", r"sample|specimen|particle|\bid\b|_id\b|^id|id$|code|label|name|\bno\b|number|#"),
]
FIELD_CATEGORIES = [(k, re.compile(p, re.I)) for k, p in FIELD_CATEGORIES]


def field_category(name):
    for cat, pattern in FIELD_CATEGORIES:
        if pattern.search(str(name)):
            return cat
    return "uncategorised"


def infer_pg_type(s):
    nn = s.dropna()
    if nn.empty:
        return "TEXT (all values missing)"
    if pd.api.types.is_bool_dtype(nn):
        return "BOOLEAN"
    if pd.api.types.is_datetime64_any_dtype(nn):
        return "DATE" if (nn.dt.normalize() == nn).all() else "TIMESTAMPTZ"
    if pd.api.types.is_numeric_dtype(nn):
        if np.all(np.mod(nn.astype(float), 1) == 0):
            return "INTEGER" if nn.abs().max() < 2**31 else "BIGINT"
        return "DOUBLE PRECISION"
    txt = nn.astype(str).str.strip()
    if txt.str.lower().isin({"yes", "no", "true", "false", "y", "n"}).all():
        return "BOOLEAN (from text)"
    if pd.to_numeric(txt.str.replace(",", ".", regex=False), errors="coerce").notna().mean() >= 0.95:
        return "DOUBLE PRECISION (currently stored as text)"
    if txt.str.match(DATE_VALUE_RE).mean() >= 0.8:
        return "DATE (currently text; verify format)"
    return f"TEXT (max length {int(txt.str.len().max())})"


metadata_fields = pd.DataFrame()
if require_data("Section 6"):
    field_rows = []
    for name, df in metadata_tables.items():
        for c in df.columns:
            s = df[c]
            n_unique = s.nunique(dropna=True)
            field_rows.append({"source": name, "source_type": "metadata table", "field": c,
                               "category": field_category(c), "postgres_type_candidate": infer_pg_type(s),
                               "pct_missing": round(100 * s.isna().mean(), 1), "n_unique": n_unique,
                               "unique_per_row": round(n_unique / max(1, s.notna().sum()), 3),
                               "candidate_key": bool(n_unique == len(s) and s.notna().all() and len(s) > 1),
                               "examples": "; ".join(map(str, s.dropna().unique()[:3]))[:100]})
    if len(spectrum_meta_long):
        emb = (spectrum_meta_long.groupby("key")
               .agg(n_spectra=("spectrum_id", "nunique"), n_files=("rel_path", "nunique"), n_unique=("value", "nunique"),
                    examples=("value", lambda v: "; ".join(v.unique()[:3])[:100])).reset_index())
        for r in emb.itertuples(index=False):
            field_rows.append({"source": f"embedded in {r.n_files} spectral file(s)", "source_type": "spectral file header",
                               "field": r.key, "category": field_category(r.key),
                               "postgres_type_candidate": infer_pg_type(spectrum_meta_long.loc[spectrum_meta_long["key"] == r.key, "value"]),
                               "pct_missing": round(100 * (1 - r.n_spectra / max(1, len(spectra_df))), 1),
                               "n_unique": r.n_unique, "unique_per_row": round(r.n_unique / max(1, r.n_spectra), 3),
                               "candidate_key": False, "examples": r.examples})
    metadata_fields = pd.DataFrame(field_rows)
    if len(metadata_fields):
        save_table(metadata_fields, "metadata_field_catalogue")
        display(Markdown("### 6.1 Metadata fields found"))
        display(metadata_fields.groupby("category").agg(n_fields=("field", "size"),
                                                         fields=("field", lambda f: ", ".join(map(str, f.unique()[:12])))))
        display(metadata_fields.head(60))
        wanted = ["sample_identifier", "sampling_location", "collection_date", "polymer_type", "particle_shape",
                  "particle_size", "instrument", "measurement_settings", "sample_preparation", "qc_annotation"]
        missing_cats = [c for c in wanted if c not in set(metadata_fields["category"])]
        print("Expected metadata categories NOT found by name:", missing_cats or "none")
    else:
        print("No metadata fields were found in non-spectral tables or spectral-file headers.")

### 6.2 Linking spectra to metadata

Identifiers on the spectrum side are the file stems plus any sample labels inside spectral files. They are compared with values in metadata columns categorised as `sample_identifier` (or with `id` in the name), after **normalisation**: lower-case, with spaces, dashes, and underscores removed. The report covers:
* **exact match rate**: the normalised IDs are equal.
* **containment match**: the metadata ID appears inside a filename, e.g. `S12` in `S12_rep2_ATR`. This is only approximate.
* **format inconsistencies**: several ID "shapes" (`A9-9` vs `A99`), case or whitespace variants, duplicate IDs, and missing IDs.

In [ ]:
def norm_id(v):
    return re.sub(r"[^a-z0-9]", "", str(v).lower())


def id_shape(v):
    return re.sub(r"[A-Za-z]+", "A", re.sub(r"\d+", "9", str(v).strip()))


linkage_report, best_link = pd.DataFrame(), None
if require_data("Section 6.2") and len(spectra_df):
    spec_ids = pd.DataFrame({"spectrum_id": spectra_df["spectrum_id"],
                             "stem": spectra_df["rel_path"].map(lambda p: Path(p).stem),
                             "label": spectra_df["label"]})
    spec_norms = set(spec_ids["stem"].map(norm_id)) | set(spec_ids["label"].dropna().map(norm_id))
    stem_norms = sorted(set(spec_ids["stem"].map(norm_id)))
    rows = []
    for name, df in metadata_tables.items():
        for c in df.columns:
            if field_category(c) != "sample_identifier" and not re.search(r"id", str(c), re.I):
                continue
            vals = df[c]
            nn = vals.dropna().astype(str)
            if nn.empty:
                continue
            norms = nn.map(norm_id)
            uniq_norm = set(norms) - {""}
            exact = {v for v in uniq_norm if v in spec_norms}
            contained = {v for v in uniq_norm - exact if len(v) >= 3 and any(v in s for s in stem_norms[:20000])}
            shapes = nn.map(id_shape).value_counts()
            rows.append({"table": name, "id_column": c, "n_rows": len(df), "n_missing_ids": int(vals.isna().sum()),
                         "n_unique_raw": nn.nunique(), "n_unique_normalised": len(uniq_norm),
                         "case_or_whitespace_variants": nn.nunique() - len(uniq_norm),
                         "n_duplicate_ids": int(norms.duplicated().sum()), "n_id_shapes": len(shapes),
                         "top_id_shapes": "; ".join(f"{k}({v})" for k, v in shapes.head(4).items()),
                         "pct_metadata_ids_exact_match": round(100 * len(exact) / max(1, len(uniq_norm)), 1),
                         "pct_metadata_ids_containment_match": round(100 * len(contained) / max(1, len(uniq_norm)), 1),
                         "n_spectra_matched_exact": int(spec_ids["stem"].map(norm_id).isin(exact).sum()
                                                        + spec_ids["label"].dropna().map(norm_id).isin(exact).sum())})
    linkage_report = pd.DataFrame(rows)
    if len(linkage_report):
        linkage_report = linkage_report.sort_values(["pct_metadata_ids_exact_match", "pct_metadata_ids_containment_match"], ascending=False)
        save_table(linkage_report, "id_linkage_report")
        display(linkage_report)
        best = linkage_report.iloc[0]
        if best["pct_metadata_ids_exact_match"] + best["pct_metadata_ids_containment_match"] > 0:
            best_link = (best["table"], best["id_column"])
            print(f"Best candidate link: {best_link[0]} -> column '{best_link[1]}'")
        else:
            print("⚠️ No metadata ID column matches any spectral filename or label. Linking will need a manual key file.")
    else:
        print("⚠️ No identifier-like columns were found in metadata tables, so spectra cannot be linked to metadata yet.")
    raw_stems = spec_ids["stem"].drop_duplicates()
    print(f"Distinct spectral file stems that collide after normalisation (case/separator variants): "
          f"{len(raw_stems) - raw_stems.map(norm_id).nunique()}")

### 6.3 Polymer comparison using linked metadata labels
This step only runs if Section 6.2 found an ID link **and** the linked table has a polymer-type column. Labels are copied from the metadata as they are. Nothing is inferred.

In [ ]:
if require_data("Section 6.3"):
    linked_labels = {}
    if best_link is not None:
        table = metadata_tables[best_link[0]]
        poly_cols = [c for c in table.columns if field_category(c) == "polymer_type"]
        if poly_cols:
            mapping = {norm_id(k): str(v) for k, v in zip(table[best_link[1]], table[poly_cols[0]])
                       if pd.notna(k) and pd.notna(v)}
            long_keys = [k for k in mapping if len(k) >= 3]
            for p in plot_sample:
                keys = [norm_id(Path(p["rel_path"]).stem)] + ([norm_id(p["label"])] if p["label"] else [])
                hit = next((mapping[k] for k in keys if k in mapping), None)
                if hit is None:
                    contained = [k for k in long_keys if k in keys[0]]
                    hit = mapping[contained[0]] if len(contained) == 1 else None  # only unambiguous matches
                if hit:
                    linked_labels[p["spectrum_id"]] = hit
            print(f"Polymer column used: '{poly_cols[0]}'. Plot-sample spectra with a linked label: {len(linked_labels)}")
            plot_label_comparison(plot_sample, linked_labels, "Spectra grouped by polymer label from linked metadata",
                                  "06_polymer_comparison_linked_metadata")
        else:
            print("The linked metadata table has no polymer-type column.")
    else:
        print("No usable spectrum↔metadata link, so the comparison was skipped. No labels were invented.")

---
## 7 · Initial data-quality assessment  ⚠️ REQUIRES DATA

All findings go into one **issue register** (`tables/qc_issues.csv`), grouped into four classes:

| Class | Meaning | Typical action |
|---|---|---|
| `confirmed_error` | The file or record is objectively broken: empty, unreadable, non-finite values, non-monotonic axis, conflicting duplicate wavenumbers, malformed lines | Fix at the source or quarantine in Silver |
| `confirmed_duplicate` | Byte-identical files or numerically identical spectra | Deduplicate by checksum and keep lineage |
| `potential_anomaly` | Possibly a real scientific signal, possibly an artefact: baseline offset or drift, low SNR, extreme intensities, non-uniform grid | **A researcher must review it.** Never auto-delete. |
| `not_assessed` | A reader was unavailable or disabled, or the file was too large | Enable the reader or export to an open format |

Thresholds are in `QC_THRESHOLDS` (Section 4) and are deliberately conservative. **No observation is deleted, and no file is changed.**

In [ ]:
def issue(cls, check, rel_path, detail, spectrum_id=None):
    return {"class": cls, "check": check, "rel_path": rel_path, "spectrum_id": spectrum_id, "detail": detail}


qc_issues = pd.DataFrame()
if require_data("Section 7"):
    rows = []
    for r in parse_log.itertuples(index=False):
        if r.status == "empty_file":
            rows.append(issue("confirmed_error", "empty file", r.rel_path, r.message or "0 bytes"))
        elif r.status in ("parse_error", "unexpected_binary_content"):
            rows.append(issue("confirmed_error", f"unreadable ({r.status})", r.rel_path, r.message))
        elif r.status in ("reader_unavailable", "reader_not_enabled", "skipped_too_large"):
            rows.append(issue("not_assessed", r.status, r.rel_path, r.message))
        elif r.status == "no_content_found" and r.category in SPECTRAL_CANDIDATE_CATEGORIES:
            rows.append(issue("confirmed_error", "no table or spectrum found", r.rel_path, r.layout))
        if r.bad_lines:
            rows.append(issue("confirmed_error", "malformed lines", r.rel_path, f"{r.bad_lines} line(s) with wrong field count"))
        if r.duplicate_header_tokens:
            rows.append(issue("confirmed_error", "duplicate column headers", r.rel_path, r.duplicate_header_tokens))
    for g in dup_contents.itertuples(index=False):
        rows.append(issue("confirmed_duplicate", "byte-identical files", g.paths.split(" | ")[0], f"{g.n_copies} copies: {g.paths}"[:500]))
    if len(spectra_df):
        s = spectra_df
        T = QC_THRESHOLDS
        for r in s.itertuples(index=False):
            rp, sid = r.rel_path, r.spectrum_id
            if r.n_valid_points < 3:
                rows.append(issue("confirmed_error", "spectrum has < 3 valid points", rp, f"{r.n_valid_points} valid of {r.n_points}", sid))
                continue
            if r.n_nonfinite_x or r.n_nonfinite_y:
                rows.append(issue("confirmed_error", "missing / non-finite values", rp, f"x: {r.n_nonfinite_x}, y: {r.n_nonfinite_y} of {r.n_points}", sid))
            if r.ordering == "non_monotonic":
                rows.append(issue("confirmed_error", "non-monotonic wavenumber axis", rp, f"{r.n_order_reversals} direction reversal(s)", sid))
            if r.n_duplicate_x_conflicting_y:
                rows.append(issue("confirmed_error", "duplicate wavenumbers with different values", rp, f"{r.n_duplicate_x_conflicting_y} wavenumber(s)", sid))
            elif r.n_duplicate_x:
                rows.append(issue("potential_anomaly", "repeated wavenumbers (identical values)", rp, f"{r.n_duplicate_x}", sid))
            if r.spacing_rel_spread > T["spacing_rel_spread"]:
                rows.append(issue("potential_anomaly", "non-uniform point spacing", rp, f"spacing {r.min_spacing:.4g}–{r.max_spacing:.4g} cm-1", sid))
            if r.value_scale == "flat":
                rows.append(issue("potential_anomaly", "flat / constant signal", rp, "no dynamic range (detector or export problem?)", sid))
            kind = next((k for k in (r.y_kind_declared, r.y_kind_heuristic) if isinstance(k, str)), "")
            if kind.startswith("absorbance"):
                if r.y_max > T["absorbance_high"]:
                    rows.append(issue("potential_anomaly", "very high absorbance (possible saturation)", rp, f"max A = {r.y_max:.3g}", sid))
                if r.y_p01 < T["absorbance_negative"]:
                    rows.append(issue("potential_anomaly", "negative absorbance (baseline offset?)", rp, f"1st percentile = {r.y_p01:.3g}", sid))
            if r.value_scale == "0-100 (percent-like)" and (r.y_max > T["percent_high"] or r.y_min < T["percent_low"]):
                rows.append(issue("potential_anomaly", "percent-scale signal outside 0–100", rp, f"range {r.y_min:.3g}–{r.y_max:.3g}", sid))
            if abs(r.edge_drift_frac) > T["edge_drift_frac"]:
                rows.append(issue("potential_anomaly", "baseline drift / sloping baseline", rp, f"end-to-end drift = {r.edge_drift_frac:+.2f} × signal range", sid))
            if np.isfinite(r.snr_est) and r.snr_est < T["low_snr"]:
                rows.append(issue("potential_anomaly", "low signal-to-noise", rp, f"SNR ≈ {r.snr_est:.1f}", sid))
        # Compare intensities only among spectra of the same (declared or heuristic) y quantity.
        y_group = s["y_kind_declared"].fillna(s["y_kind_heuristic"]).fillna("unknown")
        for kind_name, g in s.groupby(y_group):
            p99 = g["y_p99"].dropna()
            mad = 1.4826 * (p99 - p99.median()).abs().median()
            if len(p99) < 5 or not mad > 0:
                continue
            z = (g["y_p99"] - p99.median()) / mad
            for r in g[z.abs() > T["robust_z_extreme"]].itertuples(index=False):
                rows.append(issue("potential_anomaly", "extreme intensity vs. dataset", r.rel_path,
                                  f"99th pct = {r.y_p99:.4g} (robust z > {T['robust_z_extreme']} within '{kind_name}')", r.spectrum_id))
        dup_spec = s[s["xy_fingerprint"].notna() & s.duplicated("xy_fingerprint", keep=False)]
        for fp, g in dup_spec.groupby("xy_fingerprint"):
            rows.append(issue("confirmed_duplicate", "numerically identical spectra", g["rel_path"].iloc[0],
                              f"{len(g)} spectra: " + " | ".join(f"{a}#{b}" for a, b in zip(g["rel_path"], g["index_in_file"]))[:500]))
        n_grids = s["median_spacing"].round(4).nunique()
        if n_grids > 1:
            rows.append(issue("potential_anomaly", "inconsistent sampling resolution across dataset", "(dataset)",
                              f"{n_grids} distinct point spacings: {sorted(s['median_spacing'].round(4).unique())[:10]}"))
        if s["ordering"].isin(["ascending"]).any() and s["ordering"].isin(["descending"]).any():
            rows.append(issue("potential_anomaly", "mixed axis ordering across files", "(dataset)", dict(s["ordering"].value_counts())))
    for name, df in metadata_tables.items():
        for c in df.columns:
            k = column_kind(df[c])
            if k in ("mixed_numeric_text", "numeric_with_decimal_comma", "numeric_stored_as_text"):
                rows.append(issue("confirmed_error" if k == "mixed_numeric_text" else "potential_anomaly",
                                  f"metadata type inconsistency ({k})", name, f"column '{c}'"))
    for r in linkage_report.itertuples(index=False):
        if r.n_missing_ids:
            rows.append(issue("confirmed_error", "missing sample identifiers", r.table, f"column '{r.id_column}': {r.n_missing_ids} empty"))
        if r.n_duplicate_ids:
            rows.append(issue("potential_anomaly", "duplicate sample identifiers", r.table, f"column '{r.id_column}': {r.n_duplicate_ids} repeated (replicates or errors?)"))
        if r.case_or_whitespace_variants:
            rows.append(issue("confirmed_error", "inconsistent ID formatting", r.table, f"column '{r.id_column}': {r.case_or_whitespace_variants} case/separator variant(s)"))

    qc_issues = pd.DataFrame(rows, columns=["class", "check", "rel_path", "spectrum_id", "detail"])
    save_table(qc_issues, "qc_issues")
    qc_counts = qc_issues.groupby(["class", "check"]).size().rename("n").reset_index().sort_values(["class", "n"], ascending=[True, False])
    save_table(qc_counts, "qc_issue_counts")
    display(qc_counts)
    if len(qc_counts):
        fig, ax = plt.subplots(figsize=(10, 0.35 * len(qc_counts) + 1.5))
        colors = {"confirmed_error": "#C44E52", "confirmed_duplicate": "#DD8452", "potential_anomaly": "#4C72B0", "not_assessed": "#8C8C8C"}
        ax.barh([f"{c} · {k}" for c, k in zip(qc_counts["class"], qc_counts["check"])], qc_counts["n"],
                color=[colors.get(c, "grey") for c in qc_counts["class"]])
        ax.invert_yaxis()
        ax.set_xlabel("Number of issues")
        ax.set_title("Data-quality issue register (no data removed)")
        save_fig(fig, "07_qc_issue_counts")

---
## 8 · Preprocessing recommendations

The recommendations below are **generated from the observations above**. Each one states the evidence that triggered it, its scientific rationale, its risks, and the conditions under which to apply it in the **Silver** layer. **None of these steps is applied in this notebook.**

General principle: keep the raw spectrum in Bronze unchanged. Treat every preprocessing step as a versioned, parameterised transformation, so that any Gold result can be traced back and recomputed.

In [ ]:
def build_recommendations():
    s = spectra_df
    has = len(s) > 0
    issues = qc_issues["check"].value_counts() if len(qc_issues) else pd.Series(dtype=int)
    n = lambda key: int(sum(v for k, v in issues.items() if key in k))
    recs = []

    def add(step, evidence, rationale, risks, conditions, priority):
        recs.append({"step": step, "priority": priority, "observed_evidence": evidence, "rationale": rationale,
                     "risks": risks, "apply_when": conditions})

    add("Quality filtering / quarantine",
        f"{n('unreadable') + n('empty') + n('non-finite') + n('non-monotonic')} confirmed file/spectrum errors in the register",
        "Broken records bias library matching and statistics.",
        "Over-aggressive thresholds can remove genuine weak-signal particles (small or weathered microplastics).",
        "Flag records in Silver with a qc_status; exclude only confirmed errors, and review potential anomalies manually.",
        "high" if n("unreadable") + n("empty") + n("non-finite") else "medium")
    if has:
        n_grids = s["median_spacing"].round(4).nunique()
        n_orders = s["ordering"].nunique()
        add("Wavenumber alignment & interpolation to a common grid",
            f"{n_grids} distinct point spacing(s); x ranges {s['x_min'].min():.0f}–{s['x_max'].max():.0f} cm-1; orderings {dict(s['ordering'].value_counts())}",
            "Similarity scores and machine-learning features need every spectrum sampled on the same wavenumbers.",
            "Interpolating to a finer grid than the optical resolution invents apparent detail. Extrapolation beyond the measured range is invalid.",
            "Sort to one direction, interpolate (linear or cubic) only within the overlapping measured range, and choose a grid spacing no finer than the coarsest nominal resolution.",
            "high" if n_grids > 1 or n_orders > 1 else "medium")
        kinds = s["y_kind_declared"].fillna(s["y_kind_heuristic"]).value_counts()
        add("Signal-unit harmonisation (absorbance vs. transmittance/reflectance)",
            f"declared: {dict(s['y_kind_declared'].fillna('undeclared').value_counts())}; heuristic: {dict(s['y_kind_heuristic'].fillna('n/a').value_counts())}",
            "Reference libraries are usually in absorbance. Comparing %T with A directly is invalid.",
            "A = −log10(T) is valid only for transmittance as a fraction (or %T/100) measured against a proper background. It does not apply to reflectance (which needs Kubelka–Munk or log(1/R)) or to single-beam data. A wrong heuristic guess would silently corrupt the data.",
            "Convert only when the quantity is declared in metadata or confirmed by the operator, and record the conversion in lineage.",
            "high" if len(kinds) > 1 or s["y_kind_declared"].isna().any() else "low")
    add("Baseline correction",
        f"{n('baseline')} spectra flagged for baseline drift; {n('negative absorbance')} with negative absorbance",
        "Scattering from irregular particles, as well as ATR contact issues, produce sloping or curved baselines that distort peak heights and correlation scores.",
        "Asymmetric least squares and rubber-band methods can erase broad real bands (e.g., O–H near 3300 cm-1 in weathered polymers) or create artificial dips.",
        "Use one documented method (e.g., ALS with fixed λ and p) on absorbance-scale data, check visually on a stratified sample, and store parameters.",
        "high" if n("baseline") + n("negative absorbance") else "medium")
    add("Smoothing / noise reduction",
        f"{n('signal-to-noise')} spectra with estimated SNR < {QC_THRESHOLDS['low_snr']:.0f}" + (f"; median SNR ≈ {s['snr_est'].median():.0f}" if has else ""),
        "Noise lowers library-match quality (HQI) and destabilises derivative features.",
        "Savitzky–Golay windows wider than the band width broaden peaks and shift maxima, and smoothing cannot recover information that was never measured.",
        "Apply only if SNR is low, use a window narrower than the narrowest diagnostic band (≈ 5–9 points at 4 cm-1 spacing), and keep an unsmoothed copy.",
        "medium" if n("signal-to-noise") else "low")
    add("Normalisation",
        "Signal magnitudes vary between spectra (see Figure 05d)" if has else "No spectra available",
        "Particle thickness and contact area change the absolute intensity. Vector/L2 normalisation, min–max scaling, or SNV makes shapes comparable.",
        "Normalisation removes absolute-intensity information (useful for thickness/QC), and min–max scaling is sensitive to single spikes.",
        "Apply after baseline correction, store the scale factor, and use the same method as the reference library used for matching.",
        "medium")
    add("Atmospheric (CO₂ / H₂O) compensation and region masking",
        "Spectral coverage includes 2400–2280 cm-1 (CO₂) and/or water-vapour regions" if has and (s["x_min"] < 2280).any() and (s["x_max"] > 2400).any() else "Coverage not confirmed",
        "Imperfect background subtraction leaves CO₂ and water-vapour lines that do not belong to the polymer. Diamond ATR also absorbs between ~2300 and 1900 cm-1.",
        "Masking removes real information in those regions (rare for common polymers). Compensation algorithms differ between vendors.",
        "Mask 2400–1900 cm-1 for similarity scoring if ATR-diamond data is confirmed in the metadata, and document the mask in lineage.",
        "medium")
    add("ATR correction",
        "Requires instrument metadata (search the field catalogue for 'ATR'/'crystal')",
        "ATR penetration depth depends on wavenumber, which increases relative intensity at low wavenumbers compared with transmission libraries.",
        "The correction needs the crystal refractive index and angle of incidence. Applying it to transmission data is wrong.",
        "Apply only for confirmed ATR measurements matched against transmission libraries.",
        "low")
    add("Duplicate handling",
        f"{n('byte-identical')} byte-identical file group(s); {n('numerically identical')} identical-spectrum group(s)",
        "Duplicates inflate class counts and leak between training and test splits.",
        "Some 'duplicates' may be intentional replicates exported twice. Verify with lab records.",
        "Deduplicate by SHA-256 in Bronze (keep all paths) and by spectrum fingerprint in Silver (keep lineage).",
        "medium" if n("identical") else "low")
    return pd.DataFrame(recs)


if require_data("Section 8"):
    recommendations = build_recommendations()
    save_table(recommendations, "preprocessing_recommendations")
    md = ["# Preprocessing recommendations (not applied in EDA)\n"]
    for r in recommendations.itertuples(index=False):
        md.append(f"## {r.step}  _(priority: {r.priority})_\n- **Evidence:** {r.observed_evidence}\n- **Rationale:** {r.rationale}\n"
                  f"- **Risks:** {r.risks}\n- **Apply when:** {r.apply_when}\n")
    (OUT["reports"] / "preprocessing_recommendations.md").write_text("\n".join(md), encoding="utf-8")
    display(Markdown("\n".join(md)))

---
## 9 · Data-architecture readiness

> This section is **design guidance only**. It does not connect to AWS, write to S3, or create database tables.

### Proposed layers

| Layer | Contents | Format | Partitioning (S3 prefix) |
|---|---|---|---|
| **Bronze** | Original files exactly as received, plus source metadata tables | Original bytes (no conversion). The manifest is Parquet/CSV. | `s3://<bucket>/bronze/ftir/ingest_date=YYYY-MM-DD/<original relative path>`; SHA-256 stored in the manifest and S3 object metadata |
| **Silver** | Validated spectra in a standard layout with explicit units, QC flags, and versioned preprocessing outputs | **Delta Lake** (Parquet), with one row per spectrum: `spectrum_id`, `wavenumber ARRAY<DOUBLE>`, `value ARRAY<DOUBLE>`, `y_quantity`, `qc_status`, lineage columns | `y_quantity` / `instrument_id` / `ingest_date`. Avoid partitioning by sample ID, which creates too many small files. |
| **Gold** | Features (band areas, peak positions, ratios), library similarity scores, classifications, and analysis-ready records joined to sample metadata | Delta Lake tables; small extracts as Parquet for the dashboard | `analysis_version` / `polymer_class` or `site_id` |

*Why arrays rather than long format in Silver?* With 1–4 k points per spectrum, long format (one row per wavenumber) multiplies the row count by about 2,000. Arrays keep one spectrum per row, and Spark can still `explode` them when needed. Silver should store spectra on their **native** grid; the common-grid version is a separate, versioned Silver table.

### Reproducible raw → processed mapping
* `raw_file_id` = SHA-256 of the original bytes, so identical content always gets the same identifier.
* `spectrum_id` = `sha256[:16]` + `-` + zero-padded index within the file. This is the same rule used in this notebook, so the IDs in `tables/spectra_summary.csv` are already valid.
* Each Silver row records the `reader` name and version, the detected `layout`, the `preprocessing_version`, a hash of the processing parameters, and the code commit. Re-running the same versions on the same Bronze bytes must reproduce the Silver output exactly.

The next cell writes three draft artefacts: a **Bronze manifest** (from the real inventory), a **PostgreSQL DDL** draft, and a **lineage spec** containing one worked example from the data.

In [ ]:
DRAFT_DDL = """-- DRAFT schema generated by 01_FTIR_Raw_Data_Exploration.ipynb. Review with the research team before use.
-- Field names below are proposals; see tables/metadata_field_catalogue.csv for the fields actually observed.

CREATE TABLE sampling_site (
    site_id            TEXT PRIMARY KEY,
    site_name          TEXT,
    latitude           DOUBLE PRECISION CHECK (latitude BETWEEN -90 AND 90),
    longitude          DOUBLE PRECISION CHECK (longitude BETWEEN -180 AND 180),
    water_body         TEXT,
    description        TEXT
);

CREATE TABLE sample (
    sample_id          TEXT PRIMARY KEY,
    site_id            TEXT REFERENCES sampling_site(site_id),
    collection_date    DATE,
    collected_at       TIMESTAMPTZ,
    matrix             TEXT,              -- water, sediment, soil, biota ...
    depth_m            DOUBLE PRECISION,
    preparation_method TEXT,              -- e.g., H2O2 digestion, density separation (ZnCl2)
    collector          TEXT,
    notes              TEXT
);

CREATE TABLE particle (
    particle_id        TEXT PRIMARY KEY,
    sample_id          TEXT NOT NULL REFERENCES sample(sample_id),
    shape              TEXT,              -- fiber, fragment, film, foam, bead/pellet (controlled vocabulary)
    color              TEXT,
    length_um          DOUBLE PRECISION CHECK (length_um >= 0),
    width_um           DOUBLE PRECISION CHECK (width_um >= 0),
    area_um2           DOUBLE PRECISION CHECK (area_um2 >= 0),
    image_uri          TEXT
);

CREATE TABLE instrument (
    instrument_id      TEXT PRIMARY KEY,
    manufacturer       TEXT,
    model              TEXT,
    serial_number      TEXT,
    accessory          TEXT,              -- e.g., diamond ATR, microscope
    detector           TEXT
);

CREATE TABLE raw_file (
    raw_file_sha256    CHAR(64) PRIMARY KEY,
    original_rel_path  TEXT NOT NULL,
    size_bytes         BIGINT NOT NULL,
    file_format        TEXT NOT NULL,
    bronze_uri         TEXT NOT NULL,
    ingest_batch       TEXT NOT NULL,
    ingested_at        TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE spectrum (
    spectrum_id        TEXT PRIMARY KEY,  -- sha256[:16] || '-' || lpad(index_in_file, 5, '0')
    raw_file_sha256    CHAR(64) NOT NULL REFERENCES raw_file(raw_file_sha256),
    index_in_file      INTEGER NOT NULL,
    particle_id        TEXT REFERENCES particle(particle_id),
    sample_id          TEXT REFERENCES sample(sample_id),
    instrument_id      TEXT REFERENCES instrument(instrument_id),
    acquired_at        TIMESTAMPTZ,
    measurement_mode   TEXT CHECK (measurement_mode IN ('ATR', 'transmission', 'reflectance', 'unknown')),
    y_quantity         TEXT CHECK (y_quantity IN ('absorbance', 'transmittance', 'reflectance', 'single_beam', 'unknown')),
    y_quantity_source  TEXT,              -- declared | operator-confirmed | heuristic
    wavenumber_min     DOUBLE PRECISION,
    wavenumber_max     DOUBLE PRECISION,
    n_points           INTEGER,
    point_spacing_cm1  DOUBLE PRECISION,
    nominal_resolution_cm1 DOUBLE PRECISION,
    n_scans            INTEGER,
    qc_status          TEXT,
    UNIQUE (raw_file_sha256, index_in_file)
);

CREATE TABLE preprocessing_run (
    preprocessing_version TEXT PRIMARY KEY,
    steps_json         JSONB NOT NULL,    -- ordered steps with parameters
    code_commit        TEXT NOT NULL,
    created_at         TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE classification_result (
    spectrum_id        TEXT REFERENCES spectrum(spectrum_id),
    preprocessing_version TEXT REFERENCES preprocessing_run(preprocessing_version),
    library_name       TEXT NOT NULL,
    polymer_class      TEXT NOT NULL,
    similarity_score   DOUBLE PRECISION,
    rank               INTEGER NOT NULL DEFAULT 1,
    method             TEXT,
    reviewed_by        TEXT,
    PRIMARY KEY (spectrum_id, preprocessing_version, library_name, rank)
);
"""

if require_data("Section 9 artefacts") and len(inventory):
    ingest_date = datetime.now(timezone.utc).date().isoformat()
    manifest = inventory[["rel_path", "file_name", "extension", "category", "size_bytes", "sha256", "modified_utc", "is_system_file"]].copy()
    if len(parse_log):
        manifest = manifest.merge(parse_log[["rel_path", "status", "reader", "layout", "n_spectra"]], on="rel_path", how="left")
    manifest["proposed_bronze_key"] = "bronze/ftir/ingest_date=" + ingest_date + "/" + manifest["rel_path"]
    manifest["ingest_recommendation"] = np.where(manifest["is_system_file"], "exclude (system/temp file)",
                                         np.where(manifest["size_bytes"] == 0, "ingest + flag empty",
                                         np.where(manifest["sha256"].isna(), "compute checksum before ingest", "ingest as-is")))
    save_table(manifest, "draft_bronze_manifest")
    display(manifest["ingest_recommendation"].value_counts().rename("n_files"))

    ddl = DRAFT_DDL
    if len(metadata_fields):
        observed = metadata_fields[metadata_fields["source_type"] == "metadata table"]
        ddl += "\n-- Observed metadata columns (source -> field : candidate type, category):\n"
        ddl += "\n".join(f"--   {r.source} -> {r.field} : {r.postgres_type_candidate}, {r.category}" for r in observed.itertuples(index=False))
    (OUT["reports"] / "draft_postgres_schema.sql").write_text(ddl + "\n", encoding="utf-8")

    example = None
    if len(spectra_df):
        e = spectra_df.iloc[0]
        sha = inventory.set_index("rel_path").loc[e["rel_path"], "sha256"]
        example = {"raw_file_sha256": sha, "original_rel_path": e["rel_path"], "index_in_file": int(e["index_in_file"]),
                   "spectrum_id": e["spectrum_id"], "detected_layout": e["layout"],
                   "bronze_key": f"bronze/ftir/ingest_date={ingest_date}/{e['rel_path']}"}
    lineage_spec = {
        "spectrum_id_rule": "first 16 hex chars of SHA-256(raw bytes) + '-' + 5-digit index of the spectrum within the file",
        "raw_file_id_rule": "SHA-256 of original bytes (content-addressed; identical copies share one id)",
        "silver_lineage_columns": ["raw_file_sha256", "index_in_file", "reader_name", "reader_version", "detected_layout",
                                   "y_quantity", "y_quantity_source", "preprocessing_version", "params_sha256", "code_commit", "processed_at"],
        "gold_lineage_columns": ["spectrum_id", "preprocessing_version", "feature_version", "library_name", "model_version"],
        "worked_example_from_this_dataset": example,
    }
    (OUT["reports"] / "raw_to_processed_mapping_spec.json").write_text(json.dumps(lineage_spec, indent=2, default=str), encoding="utf-8")
    print("Wrote: tables/draft_bronze_manifest.csv, reports/draft_postgres_schema.sql, reports/raw_to_processed_mapping_spec.json")
else:
    (OUT["reports"] / "draft_postgres_schema.sql").write_text(DRAFT_DDL, encoding="utf-8")
    print("Data not available: wrote only the generic DDL draft (reports/draft_postgres_schema.sql).")

---
## 10 · Final research summary

This summary is generated automatically from the results above and saved to `reports/EDA_SUMMARY.md` and `reports/eda_summary.json`. Every number comes from the files that were examined. **If the data was not available, the summary says so and contains no figures.**

In [ ]:
def build_summary():
    if not DATA_AVAILABLE:
        return {"data_available": False}, ("# FTIR EDA summary\n\n**Data was not accessible** at `" + str(DATA_ROOT) +
                                           "`. No results were produced. Set `DATA_ROOT` (Section 1.3) and re-run.\n")
    pl = parse_log if len(parse_log) else pd.DataFrame(columns=["status", "category"])
    cand = pl[pl["category"].isin(SPECTRAL_CANDIDATE_CATEGORIES)] if len(pl) else pl
    unreadable = cand["status"].isin(["parse_error", "unexpected_binary_content", "empty_file", "no_content_found"]).sum()
    not_assessed = cand["status"].isin(["reader_unavailable", "reader_not_enabled", "skipped_too_large"]).sum()
    s = spectra_df
    qc = qc_issues["class"].value_counts().to_dict() if len(qc_issues) else {}
    cats_found = sorted(set(metadata_fields["category"])) if len(metadata_fields) else []
    best_pct = float(linkage_report.iloc[0]["pct_metadata_ids_exact_match"]) if len(linkage_report) else 0.0
    checksum_ok = inventory["sha256"].notna().all() if len(inventory) else False
    if not len(inventory):
        readiness = "NOT READY: no files found"
    elif checksum_ok and inventory["stat_error"].isna().all():
        readiness = "READY for Bronze (raw bytes + checksums + manifest)" + (
            "; Silver blocked for some files (see not-assessed/unreadable)" if unreadable or not_assessed else "")
    else:
        readiness = "READY WITH CAVEATS: some files lack checksums or could not be accessed"
    summary = {
        "data_available": True, "data_root": str(DATA_ROOT), "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "random_seed": RANDOM_SEED,
        "1_files_examined": int(len(inventory)), "1_files_parsed": int(len(pl)),
        "2_formats_detected": inventory["extension"].value_counts().to_dict() if len(inventory) else {},
        "3_spectral_candidate_files": int(len(cand)),
        "3_valid_spectral_files": int((pl["status"] == "spectral").sum()) if len(pl) else 0,
        "3_unreadable_or_empty_candidate_files": int(unreadable),
        "3_not_assessed_files": int(not_assessed),
        "4_identifiable_spectra": int(len(s)),
        "5_wavenumber_min_range": [float(s["x_min"].min()), float(s["x_min"].max())] if len(s) else None,
        "5_wavenumber_max_range": [float(s["x_max"].min()), float(s["x_max"].max())] if len(s) else None,
        "5_points_per_spectrum_median": int(s["n_points"].median()) if len(s) else None,
        "5_distinct_point_spacings": int(s["median_spacing"].round(4).nunique()) if len(s) else 0,
        "6_metadata_tables": len(metadata_tables), "6_metadata_categories_found": cats_found,
        "6_best_id_link_exact_match_pct": best_pct,
        "7_qc_issue_counts_by_class": qc,
        "7_top_issues": qc_issues["check"].value_counts().head(8).to_dict() if len(qc_issues) else {},
        "8_preprocessing_priorities": recommendations.loc[recommendations["priority"] == "high", "step"].tolist() if "recommendations" in globals() else [],
        "9_bronze_readiness": readiness,
    }
    next_steps = ["Review `tables/qc_issues.csv` with the lab: confirm or dismiss each potential anomaly."]
    if not_assessed:
        next_steps.append(f"Enable readers or export open formats for {int(not_assessed)} not-assessed file(s) (see parse_log).")
    if best_pct < 90:
        next_steps.append("Create an authoritative sample/particle ID key (spreadsheet → PostgreSQL `sample`/`particle`) and a filename convention, because spectrum↔metadata linkage is incomplete.")
    if len(s) and (s["y_kind_declared"].isna().any()):
        next_steps.append("Record the measured y quantity (absorbance/transmittance/reflectance) and measurement mode (ATR/transmission) for every batch.")
    next_steps += ["Freeze the Bronze manifest (SHA-256 per file) and upload raw files unchanged to S3 Bronze.",
                   "Implement the Silver reader in Databricks using the same layout detection rules and spectrum_id rule, with unit tests on files listed in `tables/parse_log.csv`.",
                   "Prototype baseline correction and normalisation on a stratified sample, then validate against reference-library matches before running at scale."]
    summary["10_next_steps"] = next_steps

    def fmt_range(r):
        return "n/a" if r is None else f"{r[0]:.1f} – {r[1]:.1f} cm⁻¹"

    md = f"""# FTIR microplastics — raw data EDA summary

_Generated {summary['generated_utc']} from `{DATA_ROOT}` (seed {RANDOM_SEED}). All numbers are computed from the files examined; nothing is simulated._

1. **Files examined:** {summary['1_files_examined']:,} in inventory; {summary['1_files_parsed']:,} passed to readers.
2. **Formats detected:** {', '.join(f'`{k}` ({v})' for k, v in summary['2_formats_detected'].items()) or 'none'}.
3. **Spectral candidate files:** {summary['3_spectral_candidate_files']:,} → **{summary['3_valid_spectral_files']:,} valid**, {summary['3_unreadable_or_empty_candidate_files']:,} unreadable/empty, {summary['3_not_assessed_files']:,} not assessed (reader unavailable/disabled/too large).
4. **Identifiable spectra:** {summary['4_identifiable_spectra']:,} (median {summary['5_points_per_spectrum_median'] or 'n/a'} points).
5. **Wavenumber ranges:** lower limit {fmt_range(summary['5_wavenumber_min_range'])}; upper limit {fmt_range(summary['5_wavenumber_max_range'])}; {summary['5_distinct_point_spacings']} distinct point spacing(s).
6. **Metadata:** {summary['6_metadata_tables']} non-spectral table(s); categories found: {', '.join(cats_found) or 'none'}. Best ID link exact-match rate: {best_pct:.1f} %.
7. **Missing-data & consistency issues:** {', '.join(f'{k}: {v}' for k, v in qc.items()) or 'none recorded'}. Most frequent: {', '.join(f'{k} ({v})' for k, v in summary['7_top_issues'].items()) or 'n/a'}.
8. **Preprocessing challenges (high priority):** {', '.join(summary['8_preprocessing_priorities']) or 'none flagged high'}; see `reports/preprocessing_recommendations.md`.
9. **Bronze readiness:** {readiness}.
10. **Recommended next engineering steps:**
""" + "\n".join(f"    {i}. {t}" for i, t in enumerate(next_steps, 1)) + "\n"
    return summary, md


summary, summary_md = build_summary()
(OUT["reports"] / "EDA_SUMMARY.md").write_text(summary_md, encoding="utf-8")
(OUT["reports"] / "eda_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
(OUT["reports"] / "run_environment.json").write_text(json.dumps({
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
    "matplotlib": matplotlib.__version__, "in_colab": IN_COLAB, "data_root": str(DATA_ROOT),
    "output_dir": str(OUTPUT_DIR), "random_seed": RANDOM_SEED, "qc_thresholds": QC_THRESHOLDS,
    "config": {"MAX_FILES_TO_PARSE": MAX_FILES_TO_PARSE, "MAX_SPECTRA_FOR_PLOTS": MAX_SPECTRA_FOR_PLOTS,
               "HASH_ALL_FILES": HASH_ALL_FILES, "ENABLE_SPA_READER": ENABLE_SPA_READER}}, indent=2), encoding="utf-8")
display(Markdown(summary_md))

In [ ]:
logger.info("Run finished.")
print(f"Outputs in {OUTPUT_DIR}:")
for sub, d in OUT.items():
    files = sorted(f.name for f in d.iterdir() if f.is_file() and f.name != ".gitkeep")
    print(f"  {sub}/ ({len(files)})")
    for f in files:
        print(f"     {f}")